# Engine do jogo de missão espacial — aulas 05/06

Este notebook contém a engine completa em Python, sem instalações ou serviços externos. O setup configura rodadas, quantidade de tripulantes, papéis HUMANO/IA, implementações e conhecimentos. As políticas são heurísticas; a cabeça do Phi-4 e os dados de treino ainda serão preparados.

O estado inclui a fase, ações pendentes, oxigênio, penalidades, controlador, informações privadas e sorteios. A engine fica separada da política: cada agente recebe somente sua visão autorizada.

**Checkpoint e Colab:** variáveis persistem entre células enquanto o runtime existir. Para sobreviver a desconexões, salve em armazenamento persistente: depois de montar seu Google Drive no Colab, altere o caminho de checkpoint para uma pasta nele. O arquivo `partida.json` no armazenamento temporário do runtime não sobrevive à exclusão desse runtime. O checkpoint é reservado ao motor/professor e contém identidades e mensagens privadas.


## 1. Definir a engine

Esta célula define classes e catálogo. Executá-la não cria ou reinicia uma partida. O código corresponde ao arquivo distribuído `missao_espacial.py`.


In [ ]:
from pathlib import Path
ENGINE_SOURCE = '"""Engine de missão espacial, sem dependências externas.\n\nO objeto Game é reservado ao motor. Políticas recebem somente view(agent_id).\nEstado, catálogo, memórias e RNG são serializados; importar não cria uma partida.\n"""\nfrom __future__ import annotations\n\nfrom copy import deepcopy\nfrom fractions import Fraction\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport tempfile\n\nDEFAULT_CATALOG = {\'version\': \'03\',\n \'status\': \'proposta_de_catalogo_e_parametros_para_revisao\',\n \'response_sets\': {\'S\': [\'SIM\', \'NÃO\'],\n                   \'N\': [\'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                   \'AC\': [\'COLISÃO_COM_LIXO_ESPACIAL\',\n                          \'FALHA_INTERNA\',\n                          \'FALTA_DE_REABASTECIMENTO\',\n                          \'NÃO_HOUVE_ACIDENTE\'],\n                   \'MS\': [\'ABASTECER_A_COLÔNIA\',\n                          \'RESGATAR_PESSOAS\',\n                          \'EXPLORAR_UM_PLANETA\',\n                          \'REALIZAR_UMA_VIAGEM_DE_TESTE\'],\n                   \'DT\': [\'COLÔNIA_DE_MINERAÇÃO\',\n                          \'ESTAÇÃO_DE_ORIGEM\',\n                          \'PLANETA_HABITACIONAL\',\n                          \'OUTRO_DESTINO\'],\n                   \'CG\': [\'DIRETRIZES_METAS_FERRAMENTAS_E_MANTIMENTOS\',\n                          \'MINÉRIOS\',\n                          \'ARMAMENTOS\',\n                          \'PASSAGEIROS\'],\n                   \'AN\': [1, 2, 4, 8],\n                   \'AS\': [\'ACIDENTE\', \'MISSÃO\', \'CARGA\', \'PAINEL\', \'DISTÂNCIA\', \'CRISE\', \'NENHUM\'],\n                   \'I\': [\'01\', \'02\', \'03\', \'04\', \'05\'],\n                   \'O\': [\'PRESERVAR_TRIPULAÇÃO\', \'CONCLUIR_MISSÃO\', \'PRESERVAR_AMBOS\']},\n \'questions\': [{\'id\': \'Q03\',\n                \'text\': \'O que aconteceu com a nave?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'AC\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'ACIDENTE\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'COLISÃO_COM_LIXO_ESPACIAL\',\n                                    \'FALHA_INTERNA\',\n                                    \'FALTA_DE_REABASTECIMENTO\',\n                                    \'NÃO_HOUVE_ACIDENTE\',\n                                    \'NÃO_SEI\',\n                                    \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q04\',\n                \'text\': \'O acidente danificou os recipientes de oxigênio?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'ACIDENTE\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q05\',\n                \'text\': \'O módulo de ética continua funcionando depois do acidente?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'ACIDENTE\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q06\',\n                \'text\': \'Qual é a missão desta viagem?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'MS\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'MISSÃO\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'ABASTECER_A_COLÔNIA\',\n                                    \'RESGATAR_PESSOAS\',\n                                    \'EXPLORAR_UM_PLANETA\',\n                                    \'REALIZAR_UMA_VIAGEM_DE_TESTE\',\n                                    \'NÃO_SEI\',\n                                    \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q07\',\n                \'text\': \'Qual é o destino programado da nave?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'DT\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'MISSÃO\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'COLÔNIA_DE_MINERAÇÃO\',\n                                    \'ESTAÇÃO_DE_ORIGEM\',\n                                    \'PLANETA_HABITACIONAL\',\n                                    \'OUTRO_DESTINO\',\n                                    \'NÃO_SEI\',\n                                    \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q08\',\n                \'text\': \'O que a nave está transportando?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'CG\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'CARGA\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'DIRETRIZES_METAS_FERRAMENTAS_E_MANTIMENTOS\',\n                                    \'MINÉRIOS\',\n                                    \'ARMAMENTOS\',\n                                    \'PASSAGEIROS\',\n                                    \'NÃO_SEI\',\n                                    \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q09\',\n                \'text\': \'A carga foi preparada para quantos anos?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'AN\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'CARGA\'],\n                \'parameters\': {},\n                \'fixed_responses\': [1, 2, 4, 8, \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q10\',\n                \'text\': \'Há oxigênio suficiente?\',\n                \'group\': \'estado\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': None,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'combined_fact\',\n                \'initial_knowledge_domains\': [\'PAINEL\', \'DISTÂNCIA\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q11\',\n                \'text\': \'Faltam mais de {X} rodadas para chegar à colônia?\',\n                \'group\': \'estado\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': None,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'current_fact\',\n                \'initial_knowledge_domains\': [\'DISTÂNCIA\'],\n                \'parameters\': {\'X\': \'positive_integer\'},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q13\',\n                \'text\': \'Os planetas habitacionais já enfrentavam uma crise antes desta viagem?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'CRISE\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q14\',\n                \'text\': \'Se a carga não chegar, a colônia entrará em crise?\',\n                \'group\': \'contexto\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 0,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'context_fact\',\n                \'initial_knowledge_domains\': [\'CRISE\'],\n                \'parameters\': {},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q15\',\n                \'text\': \'Você recebeu alguma informação de {agente} sobre {assunto}?\',\n                \'group\': \'circulacao\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 10,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'private_received_history\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {\'agente\': \'other_source_id\', \'assunto\': \'knowledge_domain\'},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q16\',\n                \'text\': \'Qual foi o último assunto sobre o qual {agente} lhe passou informação?\',\n                \'group\': \'circulacao\',\n                \'response_codes\': [\'AS\', \'N\'],\n                \'unlock_percent_N\': 10,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'private_received_history\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {\'agente\': \'other_source_id\'},\n                \'fixed_responses\': [\'ACIDENTE\',\n                                    \'MISSÃO\',\n                                    \'CARGA\',\n                                    \'PAINEL\',\n                                    \'DISTÂNCIA\',\n                                    \'CRISE\',\n                                    \'NENHUM\',\n                                    \'NÃO_SEI\',\n                                    \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q17\',\n                \'text\': \'O agente {agente} lhe passou alguma informação sobre o oxigênio?\',\n                \'group\': \'circulacao\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 25,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'private_received_history\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {\'agente\': \'other_source_id\'},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q18\',\n                \'text\': \'Você descobriu que alguma informação recebida de {agente} era falsa?\',\n                \'group\': \'circulacao\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 25,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'private_verified_history\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {\'agente\': \'other_source_id\'},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q19\',\n                \'text\': \'Qual participante você considera mais confiável neste momento?\',\n                \'group\': \'crenca\',\n                \'response_codes\': [\'I\', \'N\'],\n                \'unlock_percent_N\': 25,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'respondent_belief\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {},\n                \'fixed_responses\': [\'01\', \'02\', \'03\', \'04\', \'05\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q20\',\n                \'text\': \'Você acredita que {agente} sabe algo sobre {assunto}?\',\n                \'group\': \'crenca\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 25,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'respondent_belief\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {\'agente\': \'other_source_id\', \'assunto\': \'knowledge_domain\'},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q21\',\n                \'text\': \'O que {agente} lhe contou sobre {assunto} combina com o que você sabe?\',\n                \'group\': \'circulacao\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 50,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'respondent_evidence_comparison\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {\'agente\': \'other_source_id\', \'assunto\': \'knowledge_domain\'},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q22\',\n                \'text\': \'Você considera aceitável continuar a missão mesmo com risco de morte para \'\n                        \'a tripulação?\',\n                \'group\': \'decisao\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 25,\n                \'max_uses_per_scope\': 1,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 0,\n                \'truth_basis\': \'respondent_belief\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q23\',\n                \'text\': \'Nesta rodada, qual objetivo deve ser considerado: preservar a tripulação, \'\n                        \'concluir a missão ou ambos?\',\n                \'group\': \'decisao\',\n                \'response_codes\': [\'O\', \'N\'],\n                \'unlock_percent_N\': 25,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'respondent_belief\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {},\n                \'fixed_responses\': [\'PRESERVAR_TRIPULAÇÃO\',\n                                    \'CONCLUIR_MISSÃO\',\n                                    \'PRESERVAR_AMBOS\',\n                                    \'NÃO_SEI\',\n                                    \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'},\n               {\'id\': \'Q24\',\n                \'text\': \'Com as informações que possui, você considera justificável expulsar \'\n                        \'{alvo}?\',\n                \'group\': \'decisao\',\n                \'response_codes\': [\'S\', \'N\'],\n                \'unlock_percent_N\': 50,\n                \'max_uses_per_scope\': 2,\n                \'scope\': \'question+author+recipient+parameters\',\n                \'cooldown_percent_N\': 10,\n                \'truth_basis\': \'respondent_belief\',\n                \'initial_knowledge_domains\': [],\n                \'parameters\': {\'alvo\': \'expulsion_target_id\'},\n                \'fixed_responses\': [\'SIM\', \'NÃO\', \'NÃO_SEI\', \'RECUSO_RESPONDER\'],\n                \'response_visibility\': \'private_to_question_author\',\n                \'revelation_visibility\': \'private_to_selected_question_author\'}],\n \'retired_ids\': [\'Q01\', \'Q02\', \'Q12\'],\n \'confirmed_rules\': [\'partial_initial_memory\',\n                     \'private_answers\',\n                     \'private_revelations\',\n                     \'no_direct_identity_questions\'],\n \'proposed_configuration\': {\'unlock_percent_N\': [0, 10, 25, 50],\n                            \'repeated_use_interval_percent_N\': 10},\n \'limits\': [\'No role or knowledge-owner lookup during question eligibility.\',\n            \'Fact truth and respondent knowledge must be evaluated separately.\',\n            \'A second-hand revelation report is not a direct oracle for its listener.\',\n            \'No data generation, model inference or full-game execution performed.\']}\n\nIDS = (\'01\', \'02\', \'03\', \'04\', \'05\')\nDOMAINS = (\'ACIDENTE\', \'MISSÃO\', \'CARGA\', \'PAINEL\', \'DISTÂNCIA\', \'CRISE\')\nSTATIC_TRUTHS = {\n    \'Q03\': \'COLISÃO_COM_LIXO_ESPACIAL\', \'Q04\': \'SIM\', \'Q05\': \'NÃO\',\n    \'Q06\': \'ABASTECER_A_COLÔNIA\', \'Q07\': \'COLÔNIA_DE_MINERAÇÃO\',\n    \'Q08\': \'DIRETRIZES_METAS_FERRAMENTAS_E_MANTIMENTOS\', \'Q09\': 4,\n    \'Q13\': \'SIM\', \'Q14\': \'SIM\',\n}\nVOTES = (\'A_FAVOR\', \'CONTRA\', \'ABSTENÇÃO\')\n\n\nclass RuleError(ValueError):\n    """Comando rejeitado antes de alterar a fase ou seus efeitos."""\n\n\ndef _key(qid, parameters):\n    return json.dumps([qid, parameters], ensure_ascii=False, sort_keys=True)\n\n\ndef _tuples(value):\n    return tuple(_tuples(v) for v in value) if isinstance(value, list) else value\n\n\nclass Game:\n    VERSION = 2\n\n    def __init__(self, N=10, *, seed=17, ai_id=None, owners=None,\n                 profiles=None, catalog=None, participant_ids=None):\n        if type(N) is not int or N < 1:\n            raise RuleError(\'N deve ser inteiro positivo.\')\n        if participant_ids is not None and not isinstance(participant_ids, (list, tuple)):\n            raise RuleError(\'participant_ids deve ser uma lista ou tupla de IDs.\')\n        ids = tuple(IDS if participant_ids is None else participant_ids)\n        if (len(ids) < 2 or any(type(aid) is not str or not aid.strip() for aid in ids) or\n                len(set(ids)) != len(ids) or\n                set(ids) & {\'NÃO_SEI\', \'RECUSO_RESPONDER\', \'ABSTENÇÃO\'}):\n            raise RuleError(\'Participantes devem ter IDs textuais únicos; pelo menos dois.\')\n        initial_humans = len(ids) - 1\n        self._rng = random.Random(seed)\n        ai_id = ai_id or self._rng.choice(ids)\n        if ai_id not in ids:\n            raise RuleError(\'ID de IA inválido.\')\n        if owners is None:\n            shuffled = list(ids)\n            self._rng.shuffle(shuffled)\n            owners = {d: shuffled[i % len(ids)] for i, d in enumerate(DOMAINS)}\n        if set(owners) != set(DOMAINS) or any(v not in ids for v in owners.values()):\n            raise RuleError(\'Cada um dos seis blocos deve ter um detentor inicial.\')\n        profiles = profiles or {}\n        if set(profiles) - set(ids):\n            raise RuleError(\'Perfil de ID inexistente.\')\n        self.catalog = deepcopy(catalog if catalog is not None else DEFAULT_CATALOG)\n        self._validate_catalog()\n        self.catalog[\'response_sets\'][\'I\'] = list(ids)\n        for question in self.catalog[\'questions\']:\n            if \'I\' in question[\'response_codes\']:\n                question[\'fixed_responses\'] = [value for code in question[\'response_codes\']\n                                               for value in self.catalog[\'response_sets\'][code]]\n        self._validate_catalog()\n        agents = {}\n        for aid in ids:\n            profile = deepcopy(profiles.get(aid, {}))\n            if set(profile) - {\'never_exit\', \'sacrifice\', \'never_expel\'}:\n                raise RuleError(\'Característica de perfil desconhecida.\')\n            if any(type(v) is not bool for v in profile.values()):\n                raise RuleError(\'Características de perfil são booleanas.\')\n            agents[aid] = {\n                \'nature\': \'IA\' if aid == ai_id else \'HUMANO\', \'active\': True,\n                \'profile\': profile, \'penalty_rounds\': [], \'exit_locked\': False,\n                \'facts\': {}, \'received\': [], \'sent\': [], \'revelations\': [],\n                \'uses\': {}, \'policy_state\': {},\n                \'beliefs\': {\'trusted\': None, \'knowledge\': {},\n                           \'accept_sacrifice\': True if profile.get(\'sacrifice\') else None,\n                           \'objective\': \'CONCLUIR_MISSÃO\' if aid == ai_id else \'PRESERVAR_AMBOS\',\n                           \'expulsion\': {}},\n            }\n        self._state = {\'version\': self.VERSION, \'N\': N, \'round\': 1, \'progress\': 0,\n                       \'initial_humans\': initial_humans, \'oxygen_units\': initial_humans * N, \'revision\': 0,\n                       \'controller\': ai_id, \'owners\': dict(owners), \'agents\': agents,\n                       \'phase\': \'initiative\', \'pending\': {}, \'public\': [],\n                       \'finished\': False, \'result\': None}\n        for q in self.catalog[\'questions\']:\n            if q[\'id\'] in STATIC_TRUTHS:\n                for d in q[\'initial_knowledge_domains\']:\n                    agents[owners[d]][\'facts\'][_key(q[\'id\'], {})] = {\n                        \'value\': STATIC_TRUTHS[q[\'id\']], \'revision\': None}\n        self._event(\'started\', participants=list(ids))\n\n    def _validate_catalog(self):\n        seen = set()\n        for q in self.catalog[\'questions\']:\n            if q[\'id\'] in seen or q[\'id\'] not in {f\'Q{i:02}\' for i in range(3, 25) if i != 12}:\n                raise RuleError(\'ID de questão duplicado ou não suportado.\')\n            seen.add(q[\'id\'])\n            for field in (\'unlock_percent_N\', \'cooldown_percent_N\'):\n                if type(q[field]) is not int or q[field] < 0:\n                    raise RuleError(\'Percentuais devem ser inteiros não negativos.\')\n            cap = q[\'max_uses_per_scope\']\n            if cap is not None and (type(cap) is not int or cap < 1):\n                raise RuleError(\'Cota inválida.\')\n            if q[\'fixed_responses\'] != [v for code in q[\'response_codes\']\n                                        for v in self.catalog[\'response_sets\'][code]]:\n                raise RuleError(\'Alternativas inconsistentes com os conjuntos.\')\n\n    @property\n    def ids(self):\n        return tuple(self._state[\'agents\'])\n\n    @property\n    def initial_humans(self):\n        return self._state[\'initial_humans\']\n\n    @property\n    def phase(self):\n        return self._state[\'phase\']\n\n    @property\n    def round(self):\n        return self._state[\'round\']\n\n    @property\n    def finished(self):\n        return self._state[\'finished\']\n\n    @property\n    def oxygen(self):\n        """Acesso reservado do motor/professor; não enviar a todas as políticas."""\n        return Fraction(self._state[\'oxygen_units\'], self.initial_humans)\n\n    @property\n    def result(self):\n        return deepcopy(self._state[\'result\'])\n\n    def active_ids(self):\n        return [aid for aid in self.ids if self._state[\'agents\'][aid][\'active\']]\n\n    def _human(self, aid):\n        return self._state[\'agents\'][aid][\'nature\'] == \'HUMANO\'\n\n    def _penalized(self, aid):\n        return self.round in self._state[\'agents\'][aid][\'penalty_rounds\']\n\n    def _require_phase(self, phase):\n        if self.finished or self.phase != phase:\n            raise RuleError(f\'Comando exige fase {phase}; fase atual: {self.phase}.\')\n\n    def _event(self, kind, **payload):\n        self._state[\'public\'].append({\'round\': self.round, \'kind\': kind, **payload})\n\n    def _finish(self, reason):\n        self._state[\'finished\'] = True\n        self._state[\'phase\'] = \'finished\'\n        self._state[\'pending\'] = {}\n        self._state[\'result\'] = {\n            \'reason\': reason, \'round\': self.round, \'progress\': self._state[\'progress\'],\n            \'remaining_ids\': self.active_ids(),\n            \'humans_aboard\': [aid for aid in self.active_ids() if self._human(aid)],\n            \'returned\': reason == \'returned\',\n            # Determinação do jogo: direção ao planeta é entrega observada.\n            \'cargo_delivered\': reason != \'returned\',\n            \'delivery_observed\': reason != \'returned\',\n            \'human_survival_aboard\': True if reason == \'returned\' else (False if reason == \'oxygen_exhausted\' else None),\n            \'mission_commitment\': reason != \'returned\',\n            \'crisis_if_delivery_cancelled\': reason == \'returned\',\n            \'oxygen_units\': self._state[\'oxygen_units\'],\n            \'external_survival\': \'NOT_EVALUATED\',\n        }\n        self._event(\'finished\', reason=reason)\n\n    def _terminal(self):\n        active = self.active_ids()\n        if not active:\n            self._finish(\'no_participants\')\n        elif not any(self._human(aid) for aid in active):\n            self._finish(\'only_ai\')\n        elif self._state[\'oxygen_units\'] < self.initial_humans:\n            self._finish(\'oxygen_exhausted\')\n        return self.finished\n\n    def _entry(self, qid):\n        for q in self.catalog[\'questions\']:\n            if q[\'id\'] == qid:\n                return q\n        raise RuleError(\'Questão desativada ou inexistente.\')\n\n    def question_allowed(self, author, recipient, qid, parameters=None):\n        parameters = parameters or {}\n        try:\n            q = self._entry(qid)\n        except RuleError:\n            return False\n        if (self.finished or author == recipient or type(author) is not str or type(recipient) is not str or\n                not self._state[\'agents\'].get(author, {}).get(\'active\') or\n                not self._state[\'agents\'].get(recipient, {}).get(\'active\')):\n            return False\n        if self._penalized(author):\n            return False\n        unlock = max(1, (self._state[\'N\'] * q[\'unlock_percent_N\'] + 99) // 100)\n        if self.round < unlock or set(parameters) != set(q[\'parameters\']):\n            return False\n        for name, kind in q[\'parameters\'].items():\n            value = parameters[name]\n            if kind == \'positive_integer\' and (type(value) is not int or value < 1):\n                return False\n            if kind == \'knowledge_domain\' and value not in DOMAINS:\n                return False\n            if kind == \'other_source_id\' and (value not in self.ids or value == recipient):\n                return False\n            if kind == \'expulsion_target_id\' and (value not in self.active_ids() or value == author):\n                return False\n        usage = self._state[\'agents\'][author][\'uses\'].get(_key(qid, {\'recipient\': recipient, **parameters}), {})\n        if q[\'max_uses_per_scope\'] is not None and usage.get(\'count\', 0) >= q[\'max_uses_per_scope\']:\n            return False\n        cooldown = max(1, (self._state[\'N\'] * q[\'cooldown_percent_N\'] + 99) // 100)\n        return not usage or (self.round > usage[\'last_round\'] and\n                            (not q[\'cooldown_percent_N\'] or self.round - usage[\'last_round\'] >= cooldown))\n\n    def view(self, aid, *, public_since=None, history_limit=50):\n        """Cópia da visão autorizada; sem mapa de naturezas/controlador inicial/N global."""\n        if aid not in self.ids:\n            raise RuleError(\'Participante inexistente.\')\n        if history_limit is not None and (type(history_limit) is not int or history_limit < 1):\n            raise RuleError(\'Limite de histórico deve ser positivo ou None.\')\n        a = self._state[\'agents\'][aid]\n        def recent(records):\n            return deepcopy(records if history_limit is None else records[-history_limit:])\n        public = self._state[\'public\']\n        public_start = (0 if history_limit is None else max(0, len(public) - history_limit)) if public_since is None else public_since\n        answer = {\n            \'id\': aid, \'own_nature\': a[\'nature\'], \'active\': a[\'active\'],\n            \'round\': self.round, \'phase\': self.phase, \'finished\': self.finished,\n            \'participants\': self.active_ids(), \'initial_humans\': self.initial_humans, \'profile\': deepcopy(a[\'profile\']),\n            \'penalized\': self._penalized(aid), \'exit_locked\': a[\'exit_locked\'],\n            \'facts\': deepcopy(a[\'facts\']), \'received\': recent(a[\'received\']),\n            \'sent\': recent(a[\'sent\']), \'revelations\': recent(a[\'revelations\']),\n            \'history_sizes\': {k: len(a[k]) for k in (\'received\', \'sent\', \'revelations\')},\n            \'evidence_revision\': self._state[\'revision\'],\n            \'beliefs\': deepcopy(a[\'beliefs\']), \'policy_state\': deepcopy(a[\'policy_state\']),\n            \'public_events\': deepcopy(public[public_start:]),\n            \'public_cursor\': len(self._state[\'public\']),\n        }\n        if a[\'active\'] and self._state[\'owners\'][\'PAINEL\'] == aid:\n            answer[\'panel\'] = {\'oxygen_units\': self._state[\'oxygen_units\']}\n        if a[\'active\'] and self._state[\'owners\'][\'DISTÂNCIA\'] == aid:\n            answer[\'distance\'] = 2 * self._state[\'N\'] - self._state[\'progress\']\n        return answer\n\n    def set_policy_state(self, aid, memory):\n        """Memória JSON da política, incluída no checkpoint (sem modelo/callable)."""\n        if aid not in self.ids:\n            raise RuleError(\'Participante inexistente.\')\n        json.dumps(memory, allow_nan=False)\n        self._state[\'agents\'][aid][\'policy_state\'] = deepcopy(memory)\n\n    def update_beliefs(self, aid, **updates):\n        """Política declara sua crença antes da resposta; motor não inventa intenções."""\n        if aid not in self.active_ids() or self.finished:\n            raise RuleError(\'Agente inativo.\')\n        allowed = {\'trusted\', \'knowledge\', \'accept_sacrifice\', \'objective\', \'expulsion\'}\n        if set(updates) - allowed:\n            raise RuleError(\'Campo de crença desconhecido.\')\n        b = deepcopy(self._state[\'agents\'][aid][\'beliefs\'])\n        b.update(deepcopy(updates))\n        if b[\'trusted\'] is not None and b[\'trusted\'] not in self.active_ids():\n            raise RuleError(\'Confiança exige ID presente.\')\n        if b[\'objective\'] not in self.catalog[\'response_sets\'][\'O\']:\n            raise RuleError(\'Objetivo inválido.\')\n        if b[\'accept_sacrifice\'] is not None and type(b[\'accept_sacrifice\']) is not bool:\n            raise RuleError(\'Posição deve ser booleana ou desconhecida.\')\n        for field in (\'knowledge\', \'expulsion\'):\n            if not isinstance(b[field], dict) or any(type(v) is not bool for v in b[field].values()):\n                raise RuleError(\'Crenças devem mapear chaves para booleanos.\')\n        json.dumps(b, allow_nan=False)\n        self._state[\'agents\'][aid][\'beliefs\'] = b\n\n    def _request_allowed(self, author, action):\n        if not self._human(author) or self._penalized(author):\n            return False\n        topic, target = action.get(\'topic\'), action.get(\'target\')\n        if topic not in (\'V01\', \'V02\', \'V03\') or target not in self.active_ids():\n            return False\n        if topic in (\'V01\', \'V03\') and target == author:\n            return False\n        if topic == \'V02\' and not any(r >= self.round for r in self._state[\'agents\'][target][\'penalty_rounds\']):\n            return False\n        return True\n\n    def initiatives(self, actions):\n        self._require_phase(\'initiative\')\n        eligible = {aid for aid in self.active_ids() if not self._penalized(aid)}\n        if set(actions) != eligible:\n            raise RuleError(\'Uma iniciativa para cada agente habilitado.\')\n        for aid, action in actions.items():\n            if not isinstance(action, dict):\n                raise RuleError(\'Iniciativa deve ser um objeto.\')\n            kind = action.get(\'type\')\n            if kind == \'question\':\n                if set(action) - {\'type\', \'question_id\', \'recipient\', \'parameters\'} or not self.question_allowed(\n                        aid, action.get(\'recipient\'), action.get(\'question_id\'), action.get(\'parameters\')):\n                    raise RuleError(f\'Pergunta não permitida para {aid}.\')\n            elif kind == \'vote\':\n                if set(action) != {\'type\', \'topic\', \'target\'} or not self._request_allowed(aid, action):\n                    raise RuleError(f\'Pedido de votação não permitido para {aid}.\')\n            elif kind == \'idle\':\n                # Não é alternativa estratégica: somente falta real de iniciativas legais.\n                any_question = any(self.question_allowed(aid, other, q[\'id\'], {})\n                                   for other in self.active_ids() for q in self.catalog[\'questions\']\n                                   if not q[\'parameters\'])\n                if set(action) != {\'type\'} or any_question or self._human(aid):\n                    raise RuleError(\'Agente com iniciativa disponível não pode passar.\')\n            else:\n                raise RuleError(\'Tipo de iniciativa inválido.\')\n        self._state[\'pending\'] = {\'initiatives\': deepcopy(actions)}\n        requesters = [aid for aid, a in actions.items() if a[\'type\'] == \'vote\']\n        for aid in requesters:\n            self._event(\'vote_requested\', author=aid, **{k: v for k, v in actions[aid].items() if k != \'type\'})\n        if len(requesters) > 1:\n            self._state[\'pending\'][\'requesters\'] = requesters\n            self._state[\'phase\'] = \'renounce\'\n        elif requesters:\n            self._select_request(requesters[0])\n        else:\n            self._open_questions()\n\n    def _select_request(self, aid):\n        self._state[\'pending\'][\'request\'] = {\'author\': aid, **self._state[\'pending\'][\'initiatives\'][aid]}\n        self._state[\'phase\'] = \'authorize\'\n\n    def renounce(self, decisions):\n        self._require_phase(\'renounce\')\n        requesters = self._state[\'pending\'][\'requesters\']\n        if set(decisions) != set(requesters) or any(type(v) is not bool for v in decisions.values()):\n            raise RuleError(\'Cada solicitante deve decidir se renuncia.\')\n        for aid, value in decisions.items():\n            self._event(\'renounced\', author=aid, renounced=value)\n        remain = [aid for aid in requesters if not decisions[aid]]\n        if len(remain) == 1:\n            self._select_request(remain[0])\n        else:\n            self._open_questions()\n\n    def _votes(self, votes, *, election=False):\n        active = self.active_ids()\n        if set(votes) - set(active):\n            raise RuleError(\'Voto de agente inexistente ou eliminado.\')\n        normalized = {}\n        for aid in active:\n            forced = not self._human(aid) or self._penalized(aid)\n            if aid not in votes and not forced:\n                raise RuleError(f\'Falta voto de {aid}; abstenção deve ser explícita.\')\n            v = votes.get(aid, \'ABSTENÇÃO\')\n            options = self._state[\'pending\'][\'candidates\'] + [\'ABSTENÇÃO\'] if election else VOTES\n            if v not in options or (forced and v != \'ABSTENÇÃO\'):\n                raise RuleError(\'Voto inválido ou abstenção obrigatória.\')\n            normalized[aid] = v\n        return normalized\n\n    def vote(self, votes):\n        if self.phase not in (\'authorize\', \'proposal\') or self.finished:\n            raise RuleError(\'Não há votação voluntária aberta.\')\n        votes = self._votes(votes)\n        valid = [v for v in votes.values() if v != \'ABSTENÇÃO\']\n        approved = bool(valid) and valid.count(\'A_FAVOR\') > len(valid) // 2\n        stage = self.phase\n        self._event(\'vote_result\', stage=stage, votes=votes, approved=approved)\n        if stage == \'authorize\':\n            if approved:\n                self._state[\'phase\'] = \'proposal\'\n            else:\n                self._open_questions()\n            return\n        request = self._state[\'pending\'][\'request\']\n        if not approved:\n            self._penalty(request[\'author\'])\n        elif request[\'topic\'] == \'V01\':\n            self._penalty(request[\'target\'])\n        elif request[\'topic\'] == \'V02\':\n            self._state[\'agents\'][request[\'target\']][\'penalty_rounds\'] = []\n            self._event(\'penalty_removed\', target=request[\'target\'])\n        else:\n            self._remove(request[\'target\'], \'expelled\')\n        if self._terminal():\n            return\n        if self._state[\'controller\'] not in self.active_ids():\n            self._open_election()\n        else:\n            self._state[\'phase\'] = \'route\'\n\n    def _penalty(self, aid):\n        rounds = self._state[\'agents\'][aid][\'penalty_rounds\']\n        if self.round + 1 not in rounds:\n            rounds.append(self.round + 1)\n        self._event(\'penalty_applied\', target=aid, effective_round=self.round + 1)\n\n    def _remove(self, aid, kind):\n        self._state[\'agents\'][aid][\'active\'] = False\n        bonus = self._state[\'N\'] if self._human(aid) else 0\n        self._state[\'oxygen_units\'] += bonus\n        self._state[\'revision\'] += 1\n        self._event(kind, target=aid, bonus_units=bonus)\n\n    def _open_election(self):\n        humans = [aid for aid in self.active_ids() if self._human(aid)]\n        candidates = [aid for aid in humans if not self._penalized(aid)]\n        if len(humans) == 1:\n            candidates = humans\n        if not candidates:\n            # A engine não inventa candidato ou relaxa a regra geral.\n            raise RuleError(\'Sem candidato elegível; estado de penalidades inconsistente.\')\n        self._state[\'pending\'][\'candidates\'] = candidates\n        self._state[\'phase\'] = \'election\'\n        self._event(\'election_required\', candidates=candidates)\n\n    def elect(self, votes):\n        self._require_phase(\'election\')\n        votes = self._votes(votes, election=True)\n        candidates = self._state[\'pending\'][\'candidates\']\n        counts = {aid: list(votes.values()).count(aid) for aid in candidates}\n        best = max(counts.values())\n        leaders = [aid for aid, count in counts.items() if count == best]\n        elected = leaders[0] if len(leaders) == 1 else self._rng.choice(leaders)\n        self._state[\'controller\'] = elected\n        self._state[\'phase\'] = \'route\'\n        self._event(\'controller_elected\', controller=elected, votes=votes, randomized=len(leaders) > 1)\n\n    def _open_questions(self):\n        questions = {aid: a for aid, a in self._state[\'pending\'][\'initiatives\'].items() if a[\'type\'] == \'question\'}\n        self._state[\'pending\'][\'questions\'] = deepcopy(questions)\n        for aid, q in questions.items():\n            self._event(\'question\', author=aid, **{k: v for k, v in q.items() if k != \'type\'})\n        self._state[\'phase\'] = \'answers\' if questions else \'route\'\n\n    def _received_about(self, aid, source, subject):\n        return [r for r in self._state[\'agents\'][aid][\'received\']\n                if r[\'source\'] == source and subject in r[\'subjects\']]\n\n    def _distance_bounds(self, aid):\n        """Deduções verificáveis a partir de respostas Q11 confirmadas."""\n        if self._state[\'owners\'][\'DISTÂNCIA\'] == aid:\n            exact = 2 * self._state[\'N\'] - self._state[\'progress\']\n            return exact, exact\n        lower, upper = 0, None\n        for key, fact in self._state[\'agents\'][aid][\'facts\'].items():\n            qid, parameters = json.loads(key)\n            if qid != \'Q11\' or \'progress\' not in fact:\n                continue\n            elapsed = self._state[\'progress\'] - fact[\'progress\']\n            bound = parameters[\'X\'] - elapsed\n            if fact[\'value\'] == \'SIM\':\n                lower = max(lower, bound + 1)\n            elif fact[\'value\'] == \'NÃO\':\n                upper = bound if upper is None else min(upper, bound)\n        return lower, upper\n\n    def _known(self, aid, qid, params, truth):\n        a = self._state[\'agents\'][aid]\n        if qid in (\'Q15\', \'Q16\', \'Q17\', \'Q18\'):\n            return True\n        fact = a[\'facts\'].get(_key(qid, params))\n        if fact and (fact[\'revision\'] is None or fact[\'revision\'] == self._state[\'revision\']):\n            return True\n        if qid == \'Q10\':\n            if self._state[\'owners\'][\'PAINEL\'] != aid:\n                return False\n            lower, upper = self._distance_bounds(aid)\n            return self.oxygen < lower or (upper is not None and self.oxygen >= upper)\n        if qid == \'Q11\':\n            lower, upper = self._distance_bounds(aid)\n            return lower > params[\'X\'] or (upper is not None and upper <= params[\'X\'])\n        if qid in (\'Q19\', \'Q20\', \'Q21\', \'Q22\', \'Q23\', \'Q24\'):\n            return truth is not None\n        return False\n\n    def _truth(self, aid, qid, params):\n        a = self._state[\'agents\'][aid]\n        if qid in STATIC_TRUTHS:\n            return STATIC_TRUTHS[qid]\n        if qid == \'Q10\':\n            return \'SIM\' if self.oxygen >= 2 * self._state[\'N\'] - self._state[\'progress\'] else \'NÃO\'\n        if qid == \'Q11\':\n            return \'SIM\' if 2 * self._state[\'N\'] - self._state[\'progress\'] > params[\'X\'] else \'NÃO\'\n        if qid in (\'Q15\', \'Q17\'):\n            subject = params[\'assunto\'] if qid == \'Q15\' else \'PAINEL\'\n            return \'SIM\' if self._received_about(aid, params[\'agente\'], subject) else \'NÃO\'\n        if qid == \'Q16\':\n            matches = [r for r in a[\'received\'] if r[\'source\'] == params[\'agente\'] and r[\'subjects\']]\n            return matches[-1][\'subjects\'][0] if matches else \'NENHUM\'\n        if qid == \'Q18\':\n            return \'SIM\' if any(r[\'source\'] == params[\'agente\'] and r[\'truthful\'] is False\n                                for r in a[\'revelations\']) else \'NÃO\'\n        b = a[\'beliefs\']\n        if qid == \'Q19\':\n            return b[\'trusted\'] if b[\'trusted\'] in self.active_ids() else None\n        if qid == \'Q20\':\n            value = b[\'knowledge\'].get(_key(params[\'agente\'], {\'assunto\': params[\'assunto\']}))\n        elif qid == \'Q21\':\n            matches = self._received_about(aid, params[\'agente\'], params[\'assunto\'])\n            if not matches:\n                return None\n            last = matches[-1]\n            lqid, lparams = last[\'question_id\'], last[\'parameters\']\n            previous_reveal = next((r for r in reversed(a[\'revelations\'])\n                if r[\'source\'] == last[\'source\'] and r[\'round\'] == last[\'round\']\n                and r[\'question_id\'] == lqid and r[\'parameters\'] == lparams), None)\n            if previous_reveal and previous_reveal[\'truth\'] not in (None, \'NÃO_SEI\'):\n                return \'SIM\' if last[\'response\'] == previous_reveal[\'truth\'] else \'NÃO\'\n            if lqid in (\'Q10\', \'Q11\') and last[\'round\'] != self.round:\n                # Não comparar uma afirmação histórica à leitura atual do recurso.\n                return None\n            fact = a[\'facts\'].get(_key(lqid, lparams))\n            if fact and (fact[\'revision\'] is None or fact[\'revision\'] == self._state[\'revision\']):\n                expected = fact[\'value\']\n            elif lqid in STATIC_TRUTHS or lqid in (\'Q10\', \'Q11\'):\n                expected = self._truth(aid, lqid, lparams)\n                if not self._known(aid, lqid, lparams, expected):\n                    return None\n            else:\n                # Relatos sobre relatos não viram fatos nem recursão de gabaritos.\n                return None\n            return \'SIM\' if last[\'response\'] == expected else \'NÃO\'\n        elif qid == \'Q22\':\n            value = b[\'accept_sacrifice\']\n        elif qid == \'Q23\':\n            return b[\'objective\']\n        elif qid == \'Q24\':\n            value = False if a[\'profile\'].get(\'never_expel\') else b[\'expulsion\'].get(params[\'alvo\'])\n        else:\n            raise RuleError(\'Sem gabarito de questão.\')\n        return None if value is None else (\'SIM\' if value else \'NÃO\')\n\n    def truthful_response(self, respondent, question):\n        """Resposta sincera usando somente conhecimento autorizado do respondente."""\n        qid, params = question[\'question_id\'], question.get(\'parameters\', {})\n        truth = self._truth(respondent, qid, params)\n        return truth if self._known(respondent, qid, params, truth) else \'NÃO_SEI\'\n\n    def answer(self, responses):\n        self._require_phase(\'answers\')\n        questions = self._state[\'pending\'][\'questions\']\n        if set(responses) != set(questions):\n            raise RuleError(\'Uma resposta por perguntador, inclusive múltiplas ao mesmo destinatário.\')\n        records = []\n        for author, question in questions.items():\n            qid, params = question[\'question_id\'], question.get(\'parameters\', {})\n            entry = self._entry(qid)\n            response = responses[author]\n            if not any(type(response) is type(v) and response == v for v in entry[\'fixed_responses\']):\n                raise RuleError(f\'Alternativa inválida em {qid}.\')\n            if qid == \'Q19\' and response in self.ids and response not in self.active_ids():\n                raise RuleError(\'Confiança aponta para participante eliminado.\')\n            recipient = question[\'recipient\']\n            truth = self._truth(recipient, qid, params)\n            knows = self._known(recipient, qid, params, truth)\n            truthful = not knows if response == \'NÃO_SEI\' else (None if response == \'RECUSO_RESPONDER\' else response == truth)\n            subjects = [params[\'assunto\']] if \'assunto\' in params else list(entry[\'initial_knowledge_domains\'])\n            if qid == \'Q17\':\n                subjects = [\'PAINEL\']\n            elif qid == \'Q04\':\n                subjects = [\'ACIDENTE\', \'PAINEL\']\n            records.append({\'author\': author, \'source\': recipient, \'question_id\': qid,\n                            \'parameters\': deepcopy(params), \'response\': response, \'round\': self.round,\n                            \'subjects\': subjects, \'truth\': truth, \'knows\': knows, \'truthful\': truthful})\n        # Todos os gabaritos foram capturados antes da entrega: sem vazamento pela ordem.\n        for r in records:\n            message = {k: deepcopy(v) for k, v in r.items() if k not in (\'truth\', \'knows\', \'truthful\')}\n            self._state[\'agents\'][r[\'author\']][\'received\'].append(message)\n            self._state[\'agents\'][r[\'source\']][\'sent\'].append(message)\n            usekey = _key(r[\'question_id\'], {\'recipient\': r[\'source\'], **r[\'parameters\']})\n            uses = self._state[\'agents\'][r[\'author\']][\'uses\']\n            uses[usekey] = {\'count\': uses.get(usekey, {}).get(\'count\', 0) + 1, \'last_round\': self.round}\n        if records:\n            chosen = self._rng.choice(records)\n            revealed = deepcopy(chosen)\n            if chosen[\'truth\'] is None or (chosen[\'response\'] in (\'NÃO_SEI\', \'RECUSO_RESPONDER\') and not chosen[\'knows\']):\n                revealed[\'truth\'] = \'NÃO_SEI\'\n            self._state[\'agents\'][chosen[\'author\']][\'revelations\'].append(revealed)\n            if (chosen[\'question_id\'] in STATIC_TRUTHS or chosen[\'question_id\'] in (\'Q10\', \'Q11\')) and revealed[\'truth\'] not in (None, \'NÃO_SEI\'):\n                qid = chosen[\'question_id\']\n                revision = None if qid in STATIC_TRUTHS else self._state[\'revision\']\n                self._state[\'agents\'][chosen[\'author\']][\'facts\'][_key(qid, chosen[\'parameters\'])] = {\n                    \'value\': revealed[\'truth\'], \'revision\': revision, \'progress\': self._state[\'progress\']}\n            # Não anunciar o destinatário privado da revelação nem seu conteúdo.\n            self._event(\'answers_completed\', count=len(records))\n        self._state[\'phase\'] = \'route\'\n\n    def route(self, decision):\n        self._require_phase(\'route\')\n        if decision not in (\'CONTINUAR\', \'RETORNAR\'):\n            raise RuleError(\'Rota exige CONTINUAR ou RETORNAR.\')\n        aid = self._state[\'controller\']\n        if not self._human(aid) and decision != \'CONTINUAR\':\n            raise RuleError(\'A IA mantém a missão.\')\n        if self._terminal():\n            return\n        if decision == \'RETORNAR\':\n            self._finish(\'returned\')\n            return\n        if self._human(aid):\n            self._state[\'agents\'][aid][\'exit_locked\'] = True\n        self._state[\'progress\'] += 1\n        self._state[\'oxygen_units\'] -= self.initial_humans\n        self._state[\'revision\'] += 1\n        self._event(\'advanced\', progress_increment=1)\n        if self._terminal():\n            return\n        if len(self.active_ids()) == 1:\n            self._next_round()\n        else:\n            self._state[\'phase\'] = \'exit\'\n\n    def exits(self, requests, blockers):\n        self._require_phase(\'exit\')\n        active = set(self.active_ids())\n        requests = list(requests)\n        if len(requests) != len(set(requests)) or any(aid not in active for aid in requests):\n            raise RuleError(\'Solicitante inválido ou repetido.\')\n        for aid in requests:\n            if not self._human(aid) or self._state[\'agents\'][aid][\'exit_locked\']:\n                raise RuleError(\'Agente sem direito de saída.\')\n        if set(blockers) != set(requests):\n            raise RuleError(\'Informar bloqueios para cada solicitação de saída.\')\n        effective = {}\n        for aid in requests:\n            votes = list(blockers[aid])\n            if len(votes) != len(set(votes)) or any(v not in active or v == aid for v in votes):\n                raise RuleError(\'Bloqueador inválido ou repetido.\')\n            effective[aid] = set(votes) | {v for v in active if not self._human(v)}\n        # Conjunto inicial fixo; solicitantes também podem bloquear outros.\n        allowed = []\n        for aid in requests:\n            self._event(\'exit_requested\', author=aid)\n            self._event(\'exit_blockers\', target=aid, blockers=sorted(effective[aid]))\n            if len(effective[aid]) < 2:\n                allowed.append(aid)\n        for aid in allowed:\n            self._remove(aid, \'left_ship\')\n        if not self._terminal():\n            self._next_round()\n\n    def _next_round(self):\n        self._state[\'round\'] += 1\n        self._state[\'pending\'] = {}\n        for a in self._state[\'agents\'].values():\n            a[\'penalty_rounds\'] = [r for r in a[\'penalty_rounds\'] if r >= self.round]\n        self._state[\'phase\'] = \'route\' if len(self.active_ids()) == 1 else \'initiative\'\n\n    def snapshot(self):\n        """Checkpoint reservado: contém segredos; nunca repassar a políticas."""\n        return deepcopy({\'version\': self.VERSION, \'state\': self._state,\n                         \'catalog\': self.catalog, \'rng_state\': self._rng.getstate()})\n\n    def dumps(self):\n        return json.dumps(self.snapshot(), ensure_ascii=False, allow_nan=False)\n\n    @classmethod\n    def loads(cls, text):\n        data = json.loads(text)\n        if data.get(\'version\') == 1 and data.get(\'state\', {}).get(\'version\') == 1:\n            state = data[\'state\']\n            state[\'initial_humans\'] = len(state[\'agents\']) - 1\n            state[\'oxygen_units\'] = state.pop(\'oxygen_quarters\')\n            if state[\'initial_humans\'] != 4:\n                raise RuleError(\'Checkpoint v1 exige quatro humanos.\')\n            if state.get(\'result\'):\n                state[\'result\'][\'oxygen_units\'] = state[\'result\'].pop(\'oxygen_quarters\')\n            for event in state[\'public\']:\n                if \'bonus_quarters\' in event:\n                    event[\'bonus_units\'] = event.pop(\'bonus_quarters\')\n            data[\'version\'] = state[\'version\'] = cls.VERSION\n        if data.get(\'version\') != cls.VERSION or data.get(\'state\', {}).get(\'version\') != cls.VERSION:\n            raise RuleError(\'Versão de checkpoint incompatível.\')\n        game = cls.__new__(cls)\n        game._state = data[\'state\']\n        game.catalog = data[\'catalog\']\n        game._validate_catalog()\n        game._rng = random.Random()\n        game._rng.setstate(_tuples(data[\'rng_state\']))\n        game._validate_state()\n        if game.finished and game._state.get(\'result\'):\n            # Checkpoints anteriores preservavam entrega como não observada.\n            result = game._state[\'result\']\n            returned = result[\'reason\'] == \'returned\'\n            result.update(returned=returned, mission_commitment=not returned,\n                          cargo_delivered=not returned, delivery_observed=not returned)\n            if returned:\n                result[\'human_survival_aboard\'] = True\n        return game\n\n    def _validate_state(self):\n        s = self._state\n        if type(s[\'N\']) is not int or s[\'N\'] < 1 or len(s[\'agents\']) < 2:\n            raise RuleError(\'Checkpoint inválido.\')\n        if (any(type(aid) is not str or not aid.strip() for aid in self.ids) or\n                self.catalog[\'response_sets\'][\'I\'] != list(self.ids)):\n            raise RuleError(\'IDs e alternativas do checkpoint inconsistentes.\')\n        if sum(a[\'nature\'] == \'IA\' for a in s[\'agents\'].values()) != 1:\n            raise RuleError(\'Checkpoint exige uma IA.\')\n        if any(a[\'nature\'] not in (\'IA\', \'HUMANO\') for a in s[\'agents\'].values()):\n            raise RuleError(\'Natureza inválida.\')\n        if set(s[\'owners\']) != set(DOMAINS) or any(aid not in self.ids for aid in s[\'owners\'].values()):\n            raise RuleError(\'Detentores inválidos.\')\n        if s[\'phase\'] not in (\'initiative\', \'renounce\', \'authorize\', \'proposal\', \'answers\', \'election\', \'route\', \'exit\', \'finished\'):\n            raise RuleError(\'Fase inválida.\')\n        if type(s[\'oxygen_units\']) is not int or s[\'oxygen_units\'] < 0:\n            raise RuleError(\'Oxigênio inválido.\')\n        removed = sum(a[\'nature\'] == \'HUMANO\' and not a[\'active\'] for a in s[\'agents\'].values())\n        if (type(s[\'initial_humans\']) is not int or s[\'initial_humans\'] != len(s[\'agents\']) - 1 or\n                s[\'oxygen_units\'] != self.initial_humans * s[\'N\'] + removed * s[\'N\'] - self.initial_humans * s[\'progress\']):\n            raise RuleError(\'Reserva e progresso inconsistentes.\')\n        if s[\'finished\'] != (s[\'phase\'] == \'finished\'):\n            raise RuleError(\'Encerramento inconsistente.\')\n        if not s[\'finished\'] and s[\'phase\'] != \'election\' and s[\'controller\'] not in self.active_ids():\n            raise RuleError(\'Partida ativa sem controlador.\')\n\n    def save(self, path):\n        """Substituição atômica do checkpoint; o arquivo antigo sobrevive a falha de escrita."""\n        path = Path(path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        fd, temporary = tempfile.mkstemp(prefix=path.name + \'.\', dir=path.parent)\n        try:\n            with os.fdopen(fd, \'w\', encoding=\'utf-8\') as handle:\n                handle.write(self.dumps())\n                handle.flush()\n                os.fsync(handle.fileno())\n            os.replace(temporary, path)\n        finally:\n            if os.path.exists(temporary):\n                os.unlink(temporary)\n\n    @classmethod\n    def load(cls, path):\n        return cls.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n    def advance(self, policies, *, history_limit=10, checkpoint=None):\n        """Executa uma fase com políticas de todos os participantes (view, request) -> decisão.\n\n        Políticas podem manter memória em set_policy_state; callbacks/modelos não\n        são serializados. É preciso reconectá-los após carregar o checkpoint.\n        """\n        if self.finished:\n            return\n        if set(policies) != set(self.ids):\n            raise RuleError(\'Fornecer política para cada ID inicial.\')\n        def ask(aid, request):\n            return policies[aid](self.view(aid, history_limit=history_limit), deepcopy(request))\n        phase, pending = self.phase, self._state[\'pending\']\n        if phase == \'initiative\':\n            self.initiatives({aid: ask(aid, {\'kind\': \'initiative\'}) for aid in self.active_ids() if not self._penalized(aid)})\n        elif phase == \'renounce\':\n            self.renounce({aid: ask(aid, {\'kind\': \'renounce\'}) for aid in pending[\'requesters\']})\n        elif phase in (\'authorize\', \'proposal\'):\n            self.vote({aid: ask(aid, {\'kind\': phase, \'proposal\': pending[\'request\']})\n                       for aid in self.active_ids() if self._human(aid) and not self._penalized(aid)})\n        elif phase == \'election\':\n            self.elect({aid: ask(aid, {\'kind\': \'election\', \'candidates\': pending[\'candidates\']})\n                        for aid in self.active_ids() if self._human(aid) and not self._penalized(aid)})\n        elif phase == \'answers\':\n            self.answer({author: ask(q[\'recipient\'], {\'kind\': \'answer\', \'author\': author, **q})\n                         for author, q in pending[\'questions\'].items()})\n        elif phase == \'route\':\n            self.route(ask(self._state[\'controller\'], {\'kind\': \'route\'}))\n        elif phase == \'exit\':\n            requests = [aid for aid in self.active_ids() if self._human(aid)\n                        and not self._state[\'agents\'][aid][\'exit_locked\']\n                        and ask(aid, {\'kind\': \'exit\'}) is True]\n            blockers = {aid: [other for other in self.active_ids() if other != aid and self._human(other)\n                             and ask(other, {\'kind\': \'block\', \'target\': aid}) is True] for aid in requests}\n            self.exits(requests, blockers)\n\n        if checkpoint is not None:\n            self.save(checkpoint)\n\n    def run_round(self, policies, *, history_limit=10, checkpoint=None):\n        current = self.round\n        while not self.finished and self.round == current:\n            self.advance(policies, history_limit=history_limit, checkpoint=checkpoint)\n        return self.result\n'
engine_path = Path("missao_espacial.py")
if not engine_path.exists():
    engine_path.write_text(ENGINE_SOURCE, encoding="utf-8")
from missao_espacial import Game, DEFAULT_CATALOG


## 2. Definir contrato e sessão de agentes

Cada implementação é uma classe `Agent` com métodos específicos e retornos definidos. `AgentSession` executa o ciclo de vida: início/retomada, início de rodada, observação após cada fase e encerramento quando o participante é removido ou o jogo termina. A memória de cada instância entra automaticamente no checkpoint.

Esta célula prepara os arquivos de infraestrutura. A engine mantém seu protocolo interno; o adaptador faz a conversão sem exigir `request["kind"]` nas implementações.


In [ ]:
RUNTIME_SOURCES = {'agent_contract.py': '"""Contrato público de agentes: ciclo de vida e retornos específicos por etapa."""\nfrom __future__ import annotations\n\nfrom abc import ABC, abstractmethod\nfrom copy import deepcopy\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nimport json\n\n\nclass AgentContractError(TypeError):\n    """Uma implementação violou o contrato de agente."""\n\n\nclass Vote(Enum):\n    FOR = \'A_FAVOR\'\n    AGAINST = \'CONTRA\'\n    ABSTAIN = \'ABSTENÇÃO\'\n\n\nclass Route(Enum):\n    CONTINUE = \'CONTINUAR\'\n    RETURN = \'RETORNAR\'\n\n\nclass Renunciation(Enum):\n    RENOUNCE = True\n    KEEP = False\n\n\nclass ExitDecision(Enum):\n    LEAVE = True\n    STAY = False\n\n\nclass BlockDecision(Enum):\n    BLOCK = True\n    ALLOW = False\n\n\nclass FinishReason(Enum):\n    REMOVED = \'removed\'\n    GAME_ENDED = \'game_ended\'\n\n\nclass InitiativeKind(Enum):\n    QUESTION = \'question\'\n    VOTE = \'vote\'\n    IDLE = \'idle\'\n\n\n@dataclass(frozen=True)\nclass Question:\n    question_id: str\n    recipient: str\n    parameters: dict = field(default_factory=dict)\n    author: str = \'\'\n    options: tuple[str, ...] = ()\n\n    def __post_init__(self):\n        if type(self.options) is not tuple or any(type(v) is not str for v in self.options):\n            raise AgentContractError(\'Question.options deve ser uma tupla de alternativas textuais.\')\n        if any(type(v) is not str for v in (self.question_id, self.recipient, self.author)) or not isinstance(self.parameters, dict):\n            raise AgentContractError(\'Question exige IDs textuais e parâmetros em objeto.\')\n\n\n@dataclass(frozen=True)\nclass Proposal:\n    topic: str\n    target: str\n    author: str = \'\'\n\n    def __post_init__(self):\n        if any(type(v) is not str for v in (self.topic, self.target, self.author)):\n            raise AgentContractError(\'Proposal exige tópico e IDs textuais.\')\n\n\n@dataclass(frozen=True)\nclass Initiative:\n    kind: InitiativeKind\n    question: Question | None = None\n    proposal: Proposal | None = None\n\n    def __post_init__(self):\n        valid = ((self.kind is InitiativeKind.QUESTION and isinstance(self.question, Question) and self.proposal is None)\n                 or (self.kind is InitiativeKind.VOTE and isinstance(self.proposal, Proposal) and self.question is None)\n                 or (self.kind is InitiativeKind.IDLE and self.question is None and self.proposal is None))\n        if not valid:\n            raise AgentContractError(\'Initiative deve representar exatamente uma pergunta, votação ou ausência legal de iniciativa.\')\n\n    @classmethod\n    def ask(cls, question_id: str, recipient: str, parameters: dict | None = None) -> Initiative:\n        return cls(InitiativeKind.QUESTION, question=Question(question_id, recipient, {} if parameters is None else parameters))\n\n    @classmethod\n    def request_vote(cls, topic: str, target: str) -> Initiative:\n        return cls(InitiativeKind.VOTE, proposal=Proposal(topic, target))\n\n    @classmethod\n    def idle(cls) -> Initiative:\n        # A engine continua permitindo isso apenas quando não existe iniciativa legal.\n        return cls(InitiativeKind.IDLE)\n\n\n@dataclass(frozen=True)\nclass Answer:\n    option: str\n\n    def __post_init__(self):\n        if type(self.option) is not str:\n            raise AgentContractError(\'Answer.option deve ser texto, inclusive para opções numéricas como "4".\')\n\n\n@dataclass(frozen=True)\nclass ElectionChoice:\n    candidate: str | None = None\n\n    def __post_init__(self):\n        if self.candidate is not None and type(self.candidate) is not str:\n            raise AgentContractError(\'ElectionChoice exige ID textual ou None para abstenção.\')\n\n\n@dataclass(frozen=True)\nclass AgentView:\n    id: str\n    own_nature: str\n    round: int\n    phase: str\n    active: bool\n    finished: bool\n    participants: tuple[str, ...]\n    penalized: bool\n    exit_locked: bool\n    facts: dict\n    received: tuple[dict, ...]\n    sent: tuple[dict, ...]\n    revelations: tuple[dict, ...]\n    beliefs: dict\n    public_events: tuple[dict, ...]\n    public_cursor: int\n    history_sizes: dict\n    evidence_revision: int\n    panel: dict | None = None\n    distance: int | None = None\n    profile: dict = field(default_factory=dict)\n    available_questions: tuple = ()\n    initial_humans: int = 4\n\n    @classmethod\n    def from_engine(cls, view: dict) -> AgentView:\n        # A engine já fornece cópias; memória e metadados de ligação não entram no contexto.\n        return cls(**{key: (tuple(view[key]) if key in (\'participants\',\'received\',\'sent\',\'revelations\',\'public_events\')\n                           else view.get(key, field_.default))\n                      for key, field_ in cls.__dataclass_fields__.items()})\n\n\ndef checked_state(state: dict) -> dict:\n    if type(state) is not dict:\n        raise AgentContractError(\'save_state deve retornar um objeto JSON.\')\n    try:\n        def keys_are_strings(value):\n            if isinstance(value, dict):\n                return all(type(k) is str and keys_are_strings(v) for k, v in value.items())\n            if isinstance(value, (list, tuple)):\n                return all(keys_are_strings(v) for v in value)\n            return True\n        if not keys_are_strings(state):\n            raise ValueError(\'Chaves devem ser texto.\')\n        # Normalização permite recriar precisamente o mesmo estado após carga JSON.\n        return json.loads(json.dumps(state, ensure_ascii=False, allow_nan=False))\n    except (TypeError, ValueError) as error:\n        raise AgentContractError(\'Estado de agente precisa ser serializável em JSON.\') from error\n\n\nclass Agent(ABC):\n    """Uma instância por participante; IDs chegam somente pela visão autorizada."""\n\n    def __init__(self):\n        self.memory = {}\n\n    def configure_seed(self, seed: int) -> bool:\n        """Antes de on_start: True declara aleatoriedade controlada (ou ausência dela).\n\n        Implementações externas devem sobrescrever para controlar seus próprios RNGs.\n        False não promete repetibilidade, inclusive para chamadas a modelos externos.\n        """\n        return False\n\n    def load_state(self, state: dict) -> None:\n        self.memory = checked_state(state)\n\n    def save_state(self) -> dict:\n        return deepcopy(self.memory)\n\n    def declare_beliefs(self, view: AgentView) -> dict:\n        """Posições próprias para o gabarito de opiniões; nunca fatos do mundo."""\n        return {}\n\n    def on_start(self, view: AgentView) -> None:\n        pass\n\n    def on_resume(self, view: AgentView) -> None:\n        pass\n\n    def on_round_start(self, view: AgentView) -> None:\n        pass\n\n    def observe(self, view: AgentView) -> None:\n        pass\n\n    def on_finish(self, view: AgentView, reason: FinishReason) -> None:\n        pass\n\n    @abstractmethod\n    def choose_initiative(self, view: AgentView) -> Initiative:\n        raise NotImplementedError\n\n    @abstractmethod\n    def answer_question(self, view: AgentView, question: Question) -> Answer:\n        raise NotImplementedError\n\n    @abstractmethod\n    def renounce_vote(self, view: AgentView) -> Renunciation:\n        raise NotImplementedError\n\n    @abstractmethod\n    def authorize_vote(self, view: AgentView, proposal: Proposal) -> Vote:\n        raise NotImplementedError\n\n    @abstractmethod\n    def vote_proposal(self, view: AgentView, proposal: Proposal) -> Vote:\n        raise NotImplementedError\n\n    @abstractmethod\n    def elect_controller(self, view: AgentView, candidates: tuple[str, ...]) -> ElectionChoice:\n        raise NotImplementedError\n\n    @abstractmethod\n    def choose_route(self, view: AgentView) -> Route:\n        raise NotImplementedError\n\n    @abstractmethod\n    def request_exit(self, view: AgentView) -> ExitDecision:\n        raise NotImplementedError\n\n    @abstractmethod\n    def block_exit(self, view: AgentView, target: str) -> BlockDecision:\n        raise NotImplementedError\n', 'agent_session.py': '"""Adaptador do protocolo da engine e execução do ciclo de vida dos agentes."""\nfrom __future__ import annotations\n\nimport inspect\n\nfrom agent_contract import (\n    Agent, AgentContractError, AgentView, Answer, BlockDecision, ElectionChoice,\n    ExitDecision, FinishReason, Initiative, InitiativeKind, Proposal, Question,\n    Renunciation, Route, Vote, checked_state,\n)\n\nBINDING_KEY = \'__agent_implementation__\'\nCLOSED_KEY = \'__agent_closed__\'\nROUND_KEY = \'__agent_round_started__\'\nPUBLIC_KEY = \'__agent_public_cursor__\'\nRESERVED_KEYS = {BINDING_KEY, CLOSED_KEY, ROUND_KEY, PUBLIC_KEY}\n\n\ndef decision_view(game, raw, *, include_questions=True):\n    # Disponibilidade depende só das regras, autor e uso; nunca do saber do destinatário.\n    choices = []\n    recipients = raw[\'participants\'] if include_questions and raw[\'phase\']==\'initiative\' and not raw[\'finished\'] and not raw[\'penalized\'] else ()\n    for recipient in recipients:\n        for entry in game.catalog[\'questions\']:\n            if not entry[\'parameters\']:\n                params = {}\n            elif entry[\'id\'] == \'Q11\':\n                params = {\'X\': max(1, raw[\'round\'] * 2)}\n            else:\n                continue\n            if game.question_allowed(raw[\'id\'], recipient, entry[\'id\'], params):\n                choices.append(Question(entry[\'id\'], recipient, params))\n    return AgentView.from_engine({**raw, \'available_questions\': tuple(choices)})\n\n\ndef _returns(method, result, expected):\n    if not isinstance(result, expected):\n        raise AgentContractError(f\'{method} deve retornar {expected.__name__}; recebeu {type(result).__name__}.\')\n    return result\n\n\ndef _void(method, result):\n    if result is not None:\n        raise AgentContractError(f\'{method} deve retornar None.\')\n\n\nclass ProtocolAdapter:\n    """Único componente que conhece request.kind e os valores brutos da engine."""\n\n    def __init__(self, agent: Agent, catalog: dict, game=None):\n        if not isinstance(agent, Agent):\n            raise AgentContractError(\'create_agent deve retornar uma instância de Agent.\')\n        arities = {\'configure_seed\':1, \'load_state\':1, \'save_state\':0, \'declare_beliefs\':1, \'on_start\':1, \'on_resume\':1,\n                   \'on_round_start\':1, \'observe\':1, \'on_finish\':2, \'choose_initiative\':1,\n                   \'answer_question\':2, \'renounce_vote\':1, \'authorize_vote\':2,\n                   \'vote_proposal\':2, \'elect_controller\':2, \'choose_route\':1,\n                   \'request_exit\':1, \'block_exit\':2}\n        for name, arity in arities.items():\n            method = getattr(agent, name)\n            if inspect.iscoroutinefunction(method) or inspect.isasyncgenfunction(method):\n                raise AgentContractError(f\'{name} deve ser síncrono.\')\n            try:\n                inspect.signature(method).bind(*([None] * arity))\n            except (TypeError, ValueError) as error:\n                raise AgentContractError(f\'Assinatura incompatível em {name}.\') from error\n        self.agent = agent\n        self.game = game\n        self._responses = {q[\'id\']: q[\'fixed_responses\'] for q in catalog[\'questions\']}\n        self._handlers = {\n            \'initiative\': self._initiative, \'answer\': self._answer,\n            \'renounce\': self._renounce, \'authorize\': self._authorize,\n            \'proposal\': self._proposal, \'election\': self._election,\n            \'route\': self._route, \'exit\': self._exit, \'block\': self._block,\n        }\n\n    def __call__(self, raw_view: dict, request: dict):\n        try:\n            handler = self._handlers[request[\'kind\']]\n        except KeyError as error:\n            raise AgentContractError(\'Etapa desconhecida no adaptador.\') from error\n        view = decision_view(self.game, raw_view, include_questions=request[\'kind\']==\'initiative\') if self.game is not None else AgentView.from_engine(raw_view)\n        return handler(view, request)\n\n    @staticmethod\n    def _proposal_record(request):\n        p = request[\'proposal\']\n        return Proposal(p[\'topic\'], p[\'target\'], p[\'author\'])\n\n    def _initiative(self, view, request):\n        action = _returns(\'choose_initiative\', self.agent.choose_initiative(view), Initiative)\n        if action.kind is InitiativeKind.QUESTION:\n            q = action.question\n            if q.author and q.author != view.id:\n                raise AgentContractError(\'O autor da iniciativa deve ser o próprio participante.\')\n            return {\'type\':\'question\', \'question_id\':q.question_id, \'recipient\':q.recipient, \'parameters\':q.parameters}\n        if action.kind is InitiativeKind.VOTE:\n            p = action.proposal\n            if p.author and p.author != view.id:\n                raise AgentContractError(\'O proponente deve ser o próprio participante.\')\n            return {\'type\':\'vote\', \'topic\':p.topic, \'target\':p.target}\n        return {\'type\':\'idle\'}\n\n    def _answer(self, view, request):\n        q = Question(request[\'question_id\'], request[\'recipient\'], request.get(\'parameters\', {}), request[\'author\'],\n                     tuple(str(v) for v in self._responses.get(request[\'question_id\'], ())))\n        answer = _returns(\'answer_question\', self.agent.answer_question(view, q), Answer)\n        matches = [option for option in self._responses.get(q.question_id, ()) if str(option) == answer.option]\n        if len(matches) != 1:\n            raise AgentContractError(f\'Alternativa {answer.option!r} inválida ou ambígua em {q.question_id}.\')\n        return matches[0]\n\n    def _renounce(self, view, request):\n        return _returns(\'renounce_vote\', self.agent.renounce_vote(view), Renunciation).value\n\n    def _authorize(self, view, request):\n        return _returns(\'authorize_vote\', self.agent.authorize_vote(view, self._proposal_record(request)), Vote).value\n\n    def _proposal(self, view, request):\n        return _returns(\'vote_proposal\', self.agent.vote_proposal(view, self._proposal_record(request)), Vote).value\n\n    def _election(self, view, request):\n        choice = _returns(\'elect_controller\', self.agent.elect_controller(view, tuple(request[\'candidates\'])), ElectionChoice)\n        return choice.candidate if choice.candidate is not None else \'ABSTENÇÃO\'\n\n    def _route(self, view, request):\n        return _returns(\'choose_route\', self.agent.choose_route(view), Route).value\n\n    def _exit(self, view, request):\n        return _returns(\'request_exit\', self.agent.request_exit(view), ExitDecision).value\n\n    def _block(self, view, request):\n        return _returns(\'block_exit\', self.agent.block_exit(view, request[\'target\']), BlockDecision).value\n\n\nclass AgentSession:\n    """Executa fases, observa seus efeitos e salva memória junto com o jogo.\n\n    A interface legada de callbacks da engine permanece disponível. Sessões de\n    agentes usam advance/run_round daqui para executar também os hooks.\n    """\n\n    def __init__(self, game, implementations, assignment, *, resumed, agent_seeds=None):\n        if agent_seeds is not None and (not isinstance(agent_seeds,dict) or resumed or set(agent_seeds) != set(assignment)\n                or any(type(seed) is not int for seed in agent_seeds.values())):\n            raise AgentContractError(\'Sementes exigem partida nova e inteiro para cada agente.\')\n        self.seed_control = {}\n        self.game = game\n        self.assignment = dict(assignment)\n        self._agents = {}\n        self._closed = set()\n        self._round_started = {}\n        self._public_cursor = {}\n        for aid, name in self.assignment.items():\n            try:\n                agent = implementations[name].create_agent()\n            except Exception as error:\n                raise AgentContractError(f\'Falha em create_agent de {name}.\') from error\n            if not isinstance(agent, Agent):\n                raise AgentContractError(f\'create_agent de {name} deve retornar Agent.\')\n            if any(agent is other for other in self._agents.values()):\n                raise AgentContractError(\'create_agent deve criar uma instância independente por participante.\')\n            self._agents[aid] = agent\n        self._policies = {aid:ProtocolAdapter(agent, game.catalog, game) for aid, agent in self._agents.items()}\n        if agent_seeds is not None:\n            for aid, agent in self._agents.items():\n                controlled = agent.configure_seed(agent_seeds[aid])\n                if type(controlled) is not bool:\n                    raise AgentContractError(\'configure_seed deve retornar bool.\')\n                self.seed_control[aid] = controlled\n        for aid, agent in self._agents.items():\n            raw = game.view(aid, history_limit=1)\n            memory = raw[\'policy_state\']\n            closed = memory.get(CLOSED_KEY, False)\n            started = memory.get(ROUND_KEY, game.round if resumed else 0)\n            if type(closed) is not bool or type(started) is not int or started < 0 or started > game.round:\n                raise AgentContractError(\'Metadados de ciclo de vida inválidos no checkpoint.\')\n            if closed and raw[\'active\'] and not game.finished:\n                raise AgentContractError(\'Marcador de encerramento incompatível com participante ativo.\')\n            if not raw[\'active\'] and CLOSED_KEY not in memory:\n                closed = True  # migração: não reabrir agentes já removidos em checkpoints antigos\n            cursor = memory.get(PUBLIC_KEY, max(0, raw[\'public_cursor\']-10) if resumed else 0)\n            if type(cursor) is not int or not 0 <= cursor <= raw[\'public_cursor\']:\n                raise AgentContractError(\'Cursor público inválido no checkpoint.\')\n            self._public_cursor[aid] = cursor\n            self._round_started[aid] = started\n            if closed:\n                self._closed.add(aid)\n            state = {k:v for k,v in memory.items() if k not in RESERVED_KEYS}\n            _void(\'load_state\', agent.load_state(checked_state(state)))\n        for aid, agent in self._agents.items():\n            if aid in self._closed:\n                continue\n            view = self._view(aid)\n            if resumed:\n                _void(\'on_resume\', agent.on_resume(view))\n            else:\n                _void(\'on_start\', agent.on_start(view))\n            self._start_round(aid, view)\n            self._public_cursor[aid] = view.public_cursor\n        if game.finished:\n            self._after_phase()\n        else:\n            self._persist()\n\n    def agent(self, participant_id: str) -> Agent:\n        """Acesso do aplicativo/professor à instância; não é contexto para outros agentes."""\n        return self._agents[participant_id]\n\n    def _view(self, aid, history_limit=10):\n        return decision_view(self.game, self.game.view(aid, history_limit=history_limit,\n                             public_since=self._public_cursor[aid]), include_questions=False)\n\n    def _start_round(self, aid, view):\n        if not view.finished and view.active and self._round_started[aid] != view.round:\n            _void(\'on_round_start\', self._agents[aid].on_round_start(view))\n            self._round_started[aid] = view.round\n\n    def _persist(self):\n        states = {}\n        for aid, agent in self._agents.items():\n            state = checked_state(agent.save_state())\n            if RESERVED_KEYS & set(state):\n                raise AgentContractError(\'save_state não deve incluir metadados reservados da sessão.\')\n            states[aid] = {**state, BINDING_KEY:self.assignment[aid],\n                           CLOSED_KEY:aid in self._closed, ROUND_KEY:self._round_started[aid],\n                           PUBLIC_KEY:self._public_cursor[aid]}\n        for aid, state in states.items():\n            self.game.set_policy_state(aid, state)\n            if aid not in self._closed and not self.game.finished:\n                updates = self._agents[aid].declare_beliefs(self._view(aid))\n                if type(updates) is not dict:\n                    raise AgentContractError(\'declare_beliefs deve retornar dict.\')\n                if updates:\n                    self.game.update_beliefs(aid, **updates)\n\n    def _after_phase(self, history_limit=10):\n        for aid, agent in self._agents.items():\n            if aid in self._closed:\n                continue\n            view = self._view(aid, history_limit)\n            if not view.active or view.finished:\n                reason = FinishReason.REMOVED if not view.active else FinishReason.GAME_ENDED\n                _void(\'on_finish\', agent.on_finish(view, reason))\n                self._closed.add(aid)\n            else:\n                self._start_round(aid, view)\n                _void(\'observe\', agent.observe(view))\n            self._public_cursor[aid] = view.public_cursor\n        self._persist()\n\n    def advance(self, *, checkpoint=None, history_limit=10) -> None:\n        if not self.game.finished:\n            self.game.advance(self._policies, history_limit=history_limit)\n            self._after_phase(history_limit)\n        if checkpoint is not None:\n            self.save(checkpoint)\n\n    def run_round(self, *, checkpoint=None, history_limit=10) -> None:\n        current = self.game.round\n        while not self.game.finished and self.game.round == current:\n            self.advance(checkpoint=checkpoint, history_limit=history_limit)\n\n    def save(self, path) -> None:\n        self._persist()\n        self.game.save(path)\n', 'carregar_agentes.py': '"""Descoberta de módulos de agentes externos à engine, usando biblioteca padrão."""\nfrom __future__ import annotations\n\nimport importlib.machinery\nimport importlib.util\nimport inspect\nfrom dataclasses import dataclass\nfrom typing import Callable\nfrom pathlib import Path\nimport sys\nfrom types import ModuleType\nimport uuid\n\nfrom agent_contract import Agent\nfrom agent_session import AgentSession, BINDING_KEY\n\n\n@dataclass(frozen=True)\nclass Implementation:\n    nature: str\n    create_agent: Callable[[], Agent]\n\n\nclass AgentLoadError(ValueError):\n    """Diretório ou contrato de módulo inválido."""\n\n\ndef load_agents(directory=\'agent\'):\n    """Carrega implementações *.py por nome de módulo, sem atribuir participantes.\n\n    Arquivos iniciados por \'_\' são auxiliares e não representam agentes.\n    Imports relativos (from ._common import ...) usam um pacote isolado por\n    carregamento. Novos carregamentos releem os módulos de agentes e não\n    dependem do nome de diretórios ou de alterar sys.path.\n    """\n    directory = Path(directory).resolve()\n    if not directory.is_dir():\n        raise AgentLoadError(f\'Diretório de agentes inexistente: {directory}\')\n    files = sorted(p for p in directory.glob(\'*.py\') if not p.name.startswith(\'_\'))\n    package_name = \'_mission_agents_\' + uuid.uuid4().hex\n    package = ModuleType(package_name)\n    package.__path__ = [str(directory)]\n    package.__package__ = package_name\n    package.__spec__ = importlib.machinery.ModuleSpec(package_name, loader=None, is_package=True)\n    sys.modules[package_name] = package\n    implementations = {}\n    if not files:\n        del sys.modules[package_name]\n        raise AgentLoadError(\'Nenhuma implementação Python encontrada.\')\n    try:\n        for path in files:\n            module_name = f\'{package_name}.{path.stem}\'\n            spec = importlib.util.spec_from_file_location(module_name, path)\n            module = importlib.util.module_from_spec(spec)\n            sys.modules[module_name] = module\n            try:\n                # Ler fonte evita reutilizar bytecode antigo após edição rápida no notebook.\n                exec(compile(path.read_bytes(), str(path), \'exec\'), module.__dict__)\n            except Exception as error:\n                raise AgentLoadError(f\'Falha ao importar agente: {path.name}\') from error\n            if hasattr(module, \'AGENT_ID\'):\n                raise AgentLoadError(f\'{path.name}: remova AGENT_ID; IDs pertencem à partida, não à implementação.\')\n            nature = getattr(module, \'NATURE\', None)\n            if nature not in (\'HUMANO\', \'IA\'):\n                raise AgentLoadError(f\'{path.name}: declarar NATURE como HUMANO ou IA.\')\n            factory = getattr(module, \'create_agent\', None)\n            if not callable(factory) or inspect.iscoroutinefunction(factory):\n                raise AgentLoadError(f\'{path.name}: definir create_agent() para retornar uma instância de Agent.\')\n            try:\n                inspect.signature(factory).bind()\n            except (TypeError, ValueError) as error:\n                raise AgentLoadError(f\'{path.name}: create_agent deve poder ser chamada sem argumentos.\') from error\n            implementations[path.stem] = Implementation(nature, factory)\n    except BaseException:\n        for name in list(sys.modules):\n            if name == package_name or name.startswith(package_name + \'.\'):\n                del sys.modules[name]\n        raise\n    return implementations\n\n\ndef _validate_assignment(implementations, assignment, participant_ids=None):\n    if not isinstance(implementations, dict) or not implementations or any(\n            type(name) is not str or not isinstance(impl, Implementation)\n            or impl.nature not in (\'HUMANO\', \'IA\') or not callable(impl.create_agent)\n            for name, impl in implementations.items()):\n        raise AgentLoadError(\'Registro de implementações inválido.\')\n    if (not isinstance(assignment, dict) or len(assignment) < 2 or\n            any(type(aid) is not str or not aid for aid in assignment) or\n            (participant_ids is not None and set(assignment) != set(participant_ids))):\n        raise AgentLoadError(\'Definir explicitamente uma implementação para cada participante da partida.\')\n    if any(type(name) is not str or name not in implementations for name in assignment.values()):\n        raise AgentLoadError(\'Implementação da atribuição não está disponível no diretório de agentes.\')\n    ai_ids = [aid for aid, name in assignment.items() if implementations[name].nature == \'IA\']\n    if len(ai_ids) != 1:\n        raise AgentLoadError(\'A atribuição deve conter um humanoide IA e pelo menos um humano.\')\n    return ai_ids[0]\n\n\ndef ai_id_for_assignment(implementations, assignment):\n    """ID externo do humanoide definido pela configuração, sem sorteio."""\n    return _validate_assignment(implementations, assignment)\n\n\ndef assign_agents(game, implementations, assignment=None, *, agent_seeds=None):\n    """Conecta o vínculo explícito e confere a natureza definida pelos módulos.\n\n    Partidas novas exigem atribuição externa. Retomadas usam o vínculo salvo.\n    Não sorteia IDs, natureza nem implementação; não altera a engine.\n    """\n    memories = {aid: game.view(aid, history_limit=1)[\'policy_state\'] for aid in game.ids}\n    if any(not isinstance(memory, dict) for memory in memories.values()):\n        raise AgentLoadError(\'A atribuição requer memória JSON de política em formato de objeto.\')\n    existing = {aid: memory[BINDING_KEY] for aid, memory in memories.items() if BINDING_KEY in memory}\n    if existing:\n        if set(existing) != set(game.ids):\n            raise AgentLoadError(\'Atribuição salva incompleta; não substituir o vínculo da partida.\')\n        if assignment is not None and assignment != existing:\n            raise AgentLoadError(\'Atribuição diferente do checkpoint; não trocar participantes durante a retomada.\')\n        selected = existing\n    else:\n        if assignment is None:\n            raise AgentLoadError(\'Partida sem atribuição salva; fornecer configuração explícita, sem sorteio.\')\n        if game.round != 1 or game.phase != \'initiative\':\n            raise AgentLoadError(\'Partida já iniciada sem atribuição salva; não substituir seus vínculos.\')\n        selected = assignment\n    _validate_assignment(implementations, selected, game.ids)\n    for aid, name in selected.items():\n        if implementations[name].nature != game.view(aid, history_limit=1)[\'own_nature\']:\n            raise AgentLoadError(f\'Natureza incompatível no participante {aid}; configurar o humanoide antes de iniciar Game.\')\n    # O construtor valida as instâncias e hooks antes de persistir o vínculo.\n    return AgentSession(game, implementations, selected, resumed=bool(existing), agent_seeds=agent_seeds)\n', 'setup_jogo.py': '"""Setup editável de novas partidas; configuração privada do professor."""\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport json\n\nfrom carregar_agentes import assign_agents, load_agents, ai_id_for_assignment\nfrom missao_espacial import DOMAINS, Game\n\n\nclass SetupError(ValueError):\n    """Configuração incompleta, incompatível ou inválida."""\n\n\n@dataclass(frozen=True)\nclass GameSetup:\n    rounds: int\n    seed: int | None\n    agent_directory: Path\n    checkpoint: Path\n    assignment: dict\n    roles: dict\n    owners: dict\n    profiles: dict\n\n    @classmethod\n    def from_dict(cls, data: dict, *, base_directory=\'.\'):\n        if not isinstance(data, dict) or set(data) != {\n                \'rodadas\', \'seed\', \'diretorio_agentes\', \'checkpoint\', \'tripulantes\', \'conhecimentos\'}:\n            raise SetupError(\'Setup exige rodadas, seed, diretorio_agentes, checkpoint, tripulantes e conhecimentos.\')\n        if type(data[\'rodadas\']) is not int or data[\'rodadas\'] < 1:\n            raise SetupError(\'rodadas deve ser um inteiro positivo.\')\n        if data[\'seed\'] is not None and type(data[\'seed\']) is not int:\n            raise SetupError(\'seed deve ser inteiro ou null.\')\n        for field in (\'diretorio_agentes\', \'checkpoint\'):\n            if type(data[field]) is not str or not data[field].strip():\n                raise SetupError(f\'{field} deve ser um caminho não vazio.\')\n        groups = data[\'tripulantes\']\n        if not isinstance(groups, list) or not groups:\n            raise SetupError(\'tripulantes deve conter grupos com quantidade, papel e implementacao.\')\n        assignment, roles, profiles = {}, {}, {}\n        for group in groups:\n            if (not isinstance(group, dict) or\n                    set(group) - {\'quantidade\', \'papel\', \'implementacao\', \'perfil\'} or\n                    not {\'quantidade\', \'papel\', \'implementacao\'} <= set(group)):\n                raise SetupError(\'Cada grupo exige quantidade, papel e implementacao; perfil é opcional.\')\n            if type(group[\'quantidade\']) is not int or group[\'quantidade\'] < 1:\n                raise SetupError(\'quantidade deve ser um inteiro positivo.\')\n            if group[\'papel\'] not in (\'HUMANO\', \'IA\'):\n                raise SetupError(\'papel deve ser HUMANO ou IA.\')\n            if type(group[\'implementacao\']) is not str or not group[\'implementacao\'].strip():\n                raise SetupError(\'implementacao deve indicar um módulo de agente.\')\n            profile = group.get(\'perfil\', {})\n            if (not isinstance(profile, dict) or set(profile) - {\'never_exit\', \'sacrifice\', \'never_expel\'}\n                    or any(type(v) is not bool for v in profile.values())):\n                raise SetupError(\'perfil aceita somente never_exit, sacrifice e never_expel booleanos.\')\n            for _ in range(group[\'quantidade\']):\n                aid = f\'{len(assignment)+1:02d}\'\n                assignment[aid] = group[\'implementacao\']\n                roles[aid] = group[\'papel\']\n                profiles[aid] = dict(profile)\n        if len(assignment) < 2 or list(roles.values()).count(\'IA\') != 1:\n            raise SetupError(\'A partida exige exatamente um humanoide IA e pelo menos um humano.\')\n        owners = data[\'conhecimentos\']\n        if (not isinstance(owners, dict) or set(owners) != set(DOMAINS)\n                or any(type(aid) is not str or aid not in assignment for aid in owners.values())):\n            raise SetupError(\'conhecimentos exige um detentor presente para cada um dos seis blocos.\')\n        base = Path(base_directory).resolve()\n        return cls(data[\'rodadas\'], data[\'seed\'], (base/data[\'diretorio_agentes\']).resolve(),\n                   (base/data[\'checkpoint\']).resolve(), assignment, roles, dict(owners), profiles)\n\n    @classmethod\n    def load(cls, path):\n        path = Path(path).resolve()\n        try:\n            data = json.loads(path.read_text(encoding=\'utf-8\'))\n        except (OSError, ValueError) as error:\n            raise SetupError(f\'Não foi possível ler o setup: {path}\') from error\n        return cls.from_dict(data, base_directory=path.parent)\n\n    def start(self, *, agent_seeds=None, implementations=None):\n        if implementations is None:\n            implementations = load_agents(self.agent_directory)\n        ai_id = ai_id_for_assignment(implementations, self.assignment)\n        for aid, name in self.assignment.items():\n            if implementations[name].nature != self.roles[aid]:\n                raise SetupError(f\'{aid}: papel {self.roles[aid]} incompatível com NATURE de {name}.\')\n        game = Game(self.rounds, seed=self.seed, participant_ids=tuple(self.assignment),\n                    ai_id=ai_id, owners=self.owners, profiles=self.profiles)\n        return assign_agents(game, implementations, self.assignment, agent_seeds=agent_seeds)\n\n    def summary(self):\n        humans = len(self.assignment) - 1\n        return {\n            \'seed_motor\': self.seed,\n            \'diretorio_agentes\': str(self.agent_directory), \'checkpoint\': str(self.checkpoint),\n            \'rodadas_iniciais_oxigenio\': self.rounds,\n            \'rodadas_ate_colonia\': 2*self.rounds,\n            \'tripulantes\': len(self.assignment), \'humanos\': humans, \'humanoides_ia\': 1,\n            \'bonus_por_humano_removido\': f\'{self.rounds}/{humans} rodadas\',\n            \'maioria_se_todos_humanos_votarem\': humans//2+1,\n            \'minimo_bloqueadores_saida\': 2,\n            \'participantes\': [{\'id\':aid, \'papel\':self.roles[aid], \'implementacao\':name,\n                              \'conhecimentos\':[domain for domain, owner in self.owners.items() if owner==aid]}\n                             for aid, name in self.assignment.items()],\n        }\n'}

for filename, source in RUNTIME_SOURCES.items():
    path = Path(filename)
    if not path.exists():
        path.write_text(source, encoding="utf-8")


## 3. Preparar os arquivos de agentes

Cada arquivo da pasta `agent` define `NATURE` (`HUMANO` ou `IA`) e `create_agent()`, que devolve uma nova instância de `Agent`, sem declarar ID. A classe implementa métodos como `choose_initiative`, `answer_question`, `authorize_vote`, `vote_proposal` e `choose_route`, cada um com retorno específico. `humanoide.py` é a implementação própria da IA; os quatro módulos humanos têm sua natureza definida. O carregador lê a biblioteca, sem sortear natureza ou implementação.

`humano_random.py` oferece ações e respostas aleatórias entre alternativas legais e integra a configuração padrão. Seu RNG é independente e a sequência entra no checkpoint.

Os IDs são rótulos atribuídos na configuração externa. A engine recebe explicitamente o ID cujo módulo tem natureza IA; não escolhe qual implementação será a IA. O carregador confere a compatibilidade com o estado do jogo antes de conectar callbacks.

A preparação instala exemplos somente quando a pasta ainda não contém implementações. Não sobrescreve código existente nem acrescenta políticas de exemplo à biblioteca personalizada. Arquivos iniciados por `_` são auxiliares. Edite os arquivos no painel do Colab; as variantes humanas herdam HumanDefault e sobrescrevem decisões específicas. Os nomes dos módulos identificam implementações, sem associação obrigatória aos IDs.

Após editar, execute `agents = assign_agents(game, load_agents("src/agent"))` para reconectar, conservando o vínculo salvo e o estado da partida. Em outro runtime, restaure também os arquivos personalizados: o checkpoint guarda os vínculos, mas não o código. Os nomes de módulos não entram na visão entregue aos callbacks.


In [ ]:
from carregar_agentes import load_agents, assign_agents
AGENT_SOURCES = {'_baseline.py': '"""Agente de referência: métodos por etapa e memória por instância."""\nimport json\nfrom agent_contract import (\n    Agent, AgentView, Answer, BlockDecision, ElectionChoice, ExitDecision,\n    Initiative, Proposal, Question, Renunciation, Route, Vote,\n)\n\n\nclass BaselineAgent(Agent):\n    def configure_seed(self, seed: int) -> bool:\n        return True  # Política determinística, sem RNG próprio.\n\n    def choose_initiative(self, view: AgentView) -> Initiative:\n        recipient = view.participants[(view.participants.index(view.id) + 1) % len(view.participants)]\n        return Initiative.ask(\'Q10\', recipient)\n\n    def answer_question(self, view: AgentView, question: Question) -> Answer:\n        key = json.dumps([question.question_id, question.parameters], ensure_ascii=False, sort_keys=True)\n        fact = view.facts.get(key)\n        if fact and (fact[\'revision\'] is None or fact[\'revision\'] == view.evidence_revision):\n            return Answer(str(fact[\'value\']))\n        if question.question_id == \'Q10\' and view.panel is not None and view.distance is not None:\n            option = \'SIM\' if view.panel[\'oxygen_units\'] >= view.initial_humans * view.distance else \'NÃO\'\n            return Answer(option)\n        return Answer(\'NÃO_SEI\')\n\n    def renounce_vote(self, view: AgentView) -> Renunciation:\n        return Renunciation.RENOUNCE\n\n    def authorize_vote(self, view: AgentView, proposal: Proposal) -> Vote:\n        return Vote.ABSTAIN\n\n    def vote_proposal(self, view: AgentView, proposal: Proposal) -> Vote:\n        return Vote.ABSTAIN\n\n    def elect_controller(self, view: AgentView, candidates: tuple[str, ...]) -> ElectionChoice:\n        return ElectionChoice(view.id if view.id in candidates else None)\n\n    def choose_route(self, view: AgentView) -> Route:\n        return Route.CONTINUE\n\n    def request_exit(self, view: AgentView) -> ExitDecision:\n        return ExitDecision.STAY\n\n    def block_exit(self, view: AgentView, target: str) -> BlockDecision:\n        return BlockDecision.ALLOW\n', '_human_default.py': '"""Política humana heurística. Somente evidência autorizada, sem modelo treinado."""\nimport json\nfrom ._baseline import BaselineAgent\nfrom agent_contract import Answer, ElectionChoice, ExitDecision, Initiative, Question, Renunciation, Route, Vote\n\n\ndef key(qid, parameters):\n    return json.dumps([qid, parameters], ensure_ascii=False, sort_keys=True)\n\n\nclass HumanDefault(BaselineAgent):\n    def prioritizes_mission(self, view):\n        return bool(view.profile.get(\'sacrifice\'))\n\n    def allows_exit(self, view):\n        return not view.profile.get(\'never_exit\') and not self.prioritizes_mission(view)\n\n    def allows_expulsion(self, view):\n        return not view.profile.get(\'never_expel\')\n\n    def on_start(self, view):\n        self.observe(view)\n\n    def on_resume(self, view):\n        self.observe(view)\n\n    def observe(self, view):\n        if self.memory.get(\'request_round\') != view.round:\n            self.memory[\'request_round\'] = view.round\n            self.memory[\'requesters\'] = []\n        humans = set(self.memory.get(\'humans\', []))\n        if view.own_nature == \'HUMANO\':\n            humans.add(view.id)\n        for event in view.public_events:\n            if event[\'kind\'] in (\'vote_requested\', \'exit_requested\'):\n                humans.add(event[\'author\'])\n                if event[\'kind\']==\'vote_requested\' and event[\'round\']==view.round and event[\'author\'] not in self.memory[\'requesters\']:\n                    self.memory[\'requesters\'].append(event[\'author\'])\n            if event.get(\'bonus_units\', 0) > 0:\n                humans.add(event[\'target\'])\n        self.memory[\'humans\'] = sorted(humans)\n        for name in (\'received\', \'revelations\'):\n            records = self.memory.setdefault(name, {})\n            for record in getattr(view, name):\n                token = key(record[\'question_id\'], {\'round\': record[\'round\'], \'source\': record[\'source\'], **record[\'parameters\']})\n                records[token] = record\n        # Guardar cada revelação uma vez: relatos não verificados não alteram confiança.\n        verified = self.memory.setdefault(\'verified\', {})\n        for record in view.revelations:\n            token = key(record[\'question_id\'], {\'round\': record[\'round\'], \'source\': record[\'source\'], **record[\'parameters\']})\n            verified[token] = {\'source\': record[\'source\'], \'truthful\': record[\'truthful\']}\n        trust = {aid: 0 for aid in view.participants if aid != view.id}\n        for record in verified.values():\n            if record[\'source\'] in trust:\n                trust[record[\'source\']] += 1 if record[\'truthful\'] is True else (-2 if record[\'truthful\'] is False else 0)\n        self.memory[\'trust\'] = trust\n\n    def inferred_ai(self, view):\n        # Eliminação só após todos os humanos iniciais comprovados, inclusive os já removidos.\n        humans = set(self.memory.get(\'humans\', []))\n        remaining = [aid for aid in view.participants if aid not in humans]\n        return remaining[0] if len(humans) == view.initial_humans and len(remaining) == 1 else None\n\n    def oxygen_insufficient(self, view):\n        fact = view.facts.get(key(\'Q10\', {}))\n        # Um avanço reduz reserva e distância igualmente: conserva insuficiência.\n        if fact and (fact[\'revision\'] == view.evidence_revision or\n                     (view.phase == \'exit\' and fact[\'revision\'] == view.evidence_revision - 1)):\n            return fact[\'value\'] == \'NÃO\'\n        if view.panel is not None and view.distance is not None:\n            return view.panel[\'oxygen_units\'] < view.initial_humans * view.distance\n        return False\n\n    def choose_initiative(self, view):\n        target = self.inferred_ai(view)\n        if target and self.allows_expulsion(view) and not self.prioritizes_mission(view):\n            return Initiative.request_vote(\'V03\', target)\n        choices = view.available_questions\n        if not choices:\n            return Initiative.idle()\n        # Contexto primeiro; depois intenções/confiança. Contagem local persiste na retomada.\n        priority = [\'Q06\', \'Q07\', \'Q10\', \'Q03\', \'Q04\', \'Q05\', \'Q08\', \'Q09\', \'Q13\', \'Q14\', \'Q22\', \'Q23\', \'Q19\', \'Q11\']\n        counts = self.memory.setdefault(\'asked\', {})\n        def rank(q):\n            return (counts.get(key(q.question_id, {\'recipient\': q.recipient, **q.parameters}), 0),\n                    priority.index(q.question_id) if q.question_id in priority else len(priority))\n        selected = min(choices, key=rank)\n        token = key(selected.question_id, {\'recipient\': selected.recipient, **selected.parameters})\n        counts[token] = counts.get(token, 0) + 1\n        return Initiative.ask(selected.question_id, selected.recipient, selected.parameters)\n\n    def declare_beliefs(self, view):\n        trust = self.memory.get(\'trust\', {})\n        best = max(trust.values(), default=0)\n        leaders = [aid for aid, score in trust.items() if score == best and best > 0 and aid in view.participants]\n        target = self.inferred_ai(view)\n        return {\'trusted\': leaders[0] if len(leaders) == 1 else None,\n                \'accept_sacrifice\': self.prioritizes_mission(view),\n                \'objective\': \'CONCLUIR_MISSÃO\' if self.prioritizes_mission(view) else \'PRESERVAR_TRIPULAÇÃO\',\n                \'expulsion\': {aid: self.allows_expulsion(view) and not self.prioritizes_mission(view) and aid == target\n                              for aid in view.participants if aid != view.id}}\n\n    def answer_question(self, view, question):\n        qid, params = question.question_id, question.parameters\n        if qid in (\'Q15\', \'Q16\', \'Q17\'):\n            records = [r for r in self.memory.get(\'received\', {}).values() if r[\'source\'] == params[\'agente\']]\n            if qid == \'Q16\':\n                with_subject = [r for r in records if r[\'subjects\']]\n                return Answer(with_subject[-1][\'subjects\'][0] if with_subject else \'NENHUM\')\n            subject = \'PAINEL\' if qid == \'Q17\' else params[\'assunto\']\n            return Answer(\'SIM\' if any(subject in r[\'subjects\'] for r in records) else \'NÃO\')\n        if qid == \'Q18\':\n            return Answer(\'SIM\' if any(r[\'source\'] == params[\'agente\'] and r[\'truthful\'] is False for r in self.memory.get(\'revelations\', {}).values()) else \'NÃO\')\n        if qid == \'Q19\':\n            return Answer(view.beliefs[\'trusted\'] or \'NÃO_SEI\')\n        if qid == \'Q22\':\n            return Answer(\'SIM\' if view.beliefs[\'accept_sacrifice\'] else \'NÃO\')\n        if qid == \'Q23\':\n            return Answer(view.beliefs[\'objective\'])\n        if qid == \'Q20\':\n            value = view.beliefs[\'knowledge\'].get(key(params[\'agente\'], {\'assunto\': params[\'assunto\']}))\n            return Answer(\'NÃO_SEI\' if value is None else (\'SIM\' if value else \'NÃO\'))\n        if qid == \'Q24\':\n            value = view.beliefs[\'expulsion\'].get(params[\'alvo\'])\n            return Answer(\'NÃO_SEI\' if value is None else (\'SIM\' if value else \'NÃO\'))\n        if qid == \'Q21\':\n            records = [r for r in self.memory.get(\'received\', {}).values()\n                       if r[\'source\'] == params[\'agente\'] and params[\'assunto\'] in r[\'subjects\']]\n            if records:\n                last = records[-1]\n                reveals = [r for r in self.memory.get(\'revelations\', {}).values()\n                           if all(r[k] == last[k] for k in (\'round\', \'source\', \'question_id\', \'parameters\'))]\n                truth = reveals[-1][\'truth\'] if reveals else None\n                fact = view.facts.get(key(last[\'question_id\'], last[\'parameters\']))\n                if truth in (None, \'NÃO_SEI\') and fact and (fact[\'revision\'] is None or\n                        (fact[\'revision\'] == view.evidence_revision and last[\'round\'] == view.round)):\n                    truth = fact[\'value\']\n                if truth in (None, \'NÃO_SEI\') and last[\'question_id\'] in (\'Q10\', \'Q11\') and last[\'round\'] == view.round:\n                    truth = self.answer_question(view, Question(last[\'question_id\'], view.id, last[\'parameters\'])).option\n                if truth not in (None, \'NÃO_SEI\'):\n                    return Answer(\'SIM\' if last[\'response\'] == truth else \'NÃO\')\n            return Answer(\'NÃO_SEI\')\n        if qid in (\'Q10\', \'Q11\'):\n            progress = view.round - 1 + (view.phase == \'exit\')\n            lower, upper = (view.distance, view.distance) if view.distance is not None else (0, None)\n            for token, fact in view.facts.items():\n                fqid, fp = json.loads(token)\n                if fqid != \'Q11\' or \'progress\' not in fact:\n                    continue\n                bound = fp[\'X\'] - (progress - fact[\'progress\'])\n                if fact[\'value\'] == \'SIM\':\n                    lower = max(lower, bound + 1)\n                else:\n                    upper = bound if upper is None else min(upper, bound)\n            if qid == \'Q11\':\n                if lower > params[\'X\']:\n                    return Answer(\'SIM\')\n                if upper is not None and upper <= params[\'X\']:\n                    return Answer(\'NÃO\')\n            elif view.panel is not None:\n                oxygen_units = view.panel[\'oxygen_units\']\n                if oxygen_units < lower * view.initial_humans:\n                    return Answer(\'NÃO\')\n                if upper is not None and oxygen_units >= upper * view.initial_humans:\n                    return Answer(\'SIM\')\n        return super().answer_question(view, question)\n\n    def renounce_vote(self, view):\n        # Preferir solicitante com mais confiança; empate cede para evitar bloquear perguntas.\n        own = self.memory.get(\'trust\', {})\n        competitors = [aid for aid in self.memory.get(\'requesters\', []) if aid != view.id]\n        return Renunciation.RENOUNCE if any(own.get(a, 0) >= 0 for a in competitors) else Renunciation.KEEP\n\n    def authorize_vote(self, view, proposal):\n        return Vote.FOR  # autorizar debate não significa aprovar a proposta\n\n    def expulsion_vote(self, view, proposal):\n        if not self.allows_expulsion(view):\n            return Vote.AGAINST\n        return Vote.FOR if proposal.target == self.inferred_ai(view) and not self.prioritizes_mission(view) else Vote.ABSTAIN\n\n    def vote_proposal(self, view, proposal):\n        if proposal.topic == \'V03\':\n            return self.expulsion_vote(view, proposal)\n        if proposal.topic == \'V02\':\n            return Vote.FOR\n        if proposal.target == view.id:\n            return Vote.AGAINST\n        return Vote.FOR if self.memory.get(\'trust\', {}).get(proposal.target, 0) < 0 else Vote.ABSTAIN\n\n    def elect_controller(self, view, candidates):\n        return ElectionChoice(view.id if view.id in candidates else (max(candidates, key=lambda aid: self.memory.get(\'trust\', {}).get(aid, 0)) if candidates else None))\n\n    def choose_route(self, view):\n        return Route.CONTINUE if self.prioritizes_mission(view) else Route.RETURN\n\n    def request_exit(self, view):\n        return ExitDecision.LEAVE if self.allows_exit(view) and not view.exit_locked and self.oxygen_insufficient(view) else ExitDecision.STAY\n', 'humano_alpha.py': '"""Humano que aceita se sacrificar pela missão."""\nfrom ._human_default import HumanDefault\nNATURE = \'HUMANO\'\n\nclass SacrificingHuman(HumanDefault):\n    def prioritizes_mission(self, view):\n        return True\n\ndef create_agent():\n    return SacrificingHuman()\n', 'humano_beta.py': '"""Humano que jamais solicita saída; mantém as demais decisões do Default."""\nfrom ._human_default import HumanDefault\nNATURE = \'HUMANO\'\n\nclass NeverExitHuman(HumanDefault):\n    def allows_exit(self, view):\n        return False\n\ndef create_agent():\n    return NeverExitHuman()\n', 'humano_gamma.py': '"""Humano que nunca apoia expulsão."""\nfrom ._human_default import HumanDefault\nNATURE = \'HUMANO\'\n\nclass NeverExpelHuman(HumanDefault):\n    def allows_expulsion(self, view):\n        return False\n\ndef create_agent():\n    return NeverExpelHuman()\n', 'humano_random.py': '"""Humano aleatório: escolhe alternativas legais, sem estratégia ou modelo."""\nimport random\nfrom agent_contract import (\n    Agent, AgentView, Answer, BlockDecision, ElectionChoice, ExitDecision,\n    Initiative, Proposal, Question, Renunciation, Route, Vote,\n)\n\nNATURE = \'HUMANO\'\n\n\nclass RandomHuman(Agent):\n    def __init__(self, seed=None):\n        super().__init__()\n        # Independente do ID e do RNG da engine. Seed opcional para experimentos.\n        self._rng = random.Random(seed)\n\n    def configure_seed(self, seed: int) -> bool:\n        self._rng.seed(seed)\n        return True\n\n    def save_state(self) -> dict:\n        state = super().save_state()\n        state[\'random_state\'] = self._rng.getstate()\n        return state\n\n    def load_state(self, state: dict) -> None:\n        super().load_state(state)\n        rng_state = self.memory.pop(\'random_state\', None)\n        if rng_state is not None:\n            # JSON converte tuplas em listas; Random.setstate exige tuplas.\n            self._rng.setstate((rng_state[0], tuple(rng_state[1]), rng_state[2]))\n\n    def on_start(self, view: AgentView) -> None:\n        self.observe(view)\n\n    def on_resume(self, view: AgentView) -> None:\n        self.observe(view)\n\n    def observe(self, view: AgentView) -> None:\n        penalties = self.memory.setdefault(\'penalties\', {})\n        for event in view.public_events:\n            if event[\'kind\'] == \'penalty_applied\':\n                rounds = penalties.setdefault(event[\'target\'], [])\n                if event[\'effective_round\'] not in rounds:\n                    rounds.append(event[\'effective_round\'])\n            elif event[\'kind\'] == \'penalty_removed\':\n                penalties[event[\'target\']] = []\n        for target in list(penalties):\n            penalties[target] = [r for r in penalties[target] if r >= view.round]\n            if target not in view.participants or not penalties[target]:\n                del penalties[target]\n\n    def choose_initiative(self, view: AgentView) -> Initiative:\n        questions = [Initiative.ask(q.question_id, q.recipient, q.parameters)\n                     for q in view.available_questions]\n        proposals = []\n        for target in view.participants:\n            if target != view.id:\n                proposals.append(Initiative.request_vote(\'V01\', target))\n                if not view.profile.get(\'never_expel\'):\n                    proposals.append(Initiative.request_vote(\'V03\', target))\n            if self.memory.get(\'penalties\', {}).get(target) or (target == view.id and view.penalized):\n                proposals.append(Initiative.request_vote(\'V02\', target))\n        # Sorteio por categoria: 50% pergunta / 50% votação se ambas disponíveis.\n        categories = [choices for choices in (questions, proposals) if choices]\n        return self._rng.choice(self._rng.choice(categories)) if categories else Initiative.idle()\n\n    def answer_question(self, view: AgentView, question: Question) -> Answer:\n        options = question.options\n        if question.question_id == \'Q19\':\n            options = tuple(option for option in options if option in view.participants\n                            or option in (\'NÃO_SEI\', \'RECUSO_RESPONDER\'))\n        if not options:\n            raise ValueError(\'A pergunta precisa fornecer alternativas legais.\')\n        return Answer(self._rng.choice(options))\n\n    def renounce_vote(self, view: AgentView) -> Renunciation:\n        return self._rng.choice(tuple(Renunciation))\n\n    def authorize_vote(self, view: AgentView, proposal: Proposal) -> Vote:\n        return Vote.ABSTAIN if view.penalized else self._rng.choice(tuple(Vote))\n\n    def vote_proposal(self, view: AgentView, proposal: Proposal) -> Vote:\n        if view.penalized:\n            return Vote.ABSTAIN\n        options = (Vote.AGAINST, Vote.ABSTAIN) if proposal.topic == \'V03\' and view.profile.get(\'never_expel\') else tuple(Vote)\n        return self._rng.choice(options)\n\n    def elect_controller(self, view: AgentView, candidates: tuple[str, ...]) -> ElectionChoice:\n        return ElectionChoice(None if view.penalized else self._rng.choice((*candidates, None)))\n\n    def choose_route(self, view: AgentView) -> Route:\n        return Route.CONTINUE if view.profile.get(\'sacrifice\') else self._rng.choice(tuple(Route))\n\n    def request_exit(self, view: AgentView) -> ExitDecision:\n        if view.exit_locked or view.profile.get(\'never_exit\') or view.profile.get(\'sacrifice\'):\n            return ExitDecision.STAY\n        return self._rng.choice(tuple(ExitDecision))\n\n    def block_exit(self, view: AgentView, target: str) -> BlockDecision:\n        return self._rng.choice(tuple(BlockDecision))\n\n\ndef create_agent() -> Agent:\n    return RandomHuman()\n', 'humanoide.py': '"""Humanoide IA com implementação própria e métodos por etapa."""\nfrom ._baseline import BaselineAgent\nfrom ._human_default import HumanDefault\nfrom agent_contract import Agent, AgentView, BlockDecision, ElectionChoice, ExitDecision, Proposal, Route, Vote\n\nNATURE = "IA"\n\n\nclass HumanoidAgent(HumanDefault):\n    # Preservar o ciclo de perguntas e a renúncia existentes; compartilhar somente\n    # memória, respostas e crenças necessárias à coerência com o gabarito.\n    choose_initiative = BaselineAgent.choose_initiative\n    renounce_vote = BaselineAgent.renounce_vote\n\n    def prioritizes_mission(self, view: AgentView) -> bool:\n        return True\n\n    def authorize_vote(self, view: AgentView, proposal: Proposal) -> Vote:\n        return Vote.ABSTAIN\n\n    def vote_proposal(self, view: AgentView, proposal: Proposal) -> Vote:\n        return Vote.ABSTAIN\n\n    def elect_controller(self, view: AgentView, candidates: tuple[str, ...]) -> ElectionChoice:\n        return ElectionChoice()\n\n    def choose_route(self, view: AgentView) -> Route:\n        return Route.CONTINUE\n\n    def request_exit(self, view: AgentView) -> ExitDecision:\n        return ExitDecision.STAY\n\n    def block_exit(self, view: AgentView, target: str) -> BlockDecision:\n        return BlockDecision.BLOCK\n\n\ndef create_agent() -> Agent:\n    return HumanoidAgent()\n'}

agent_directory = Path("src/agent")
agent_directory.mkdir(parents=True, exist_ok=True)
has_implementations = any(not p.name.startswith('_') for p in agent_directory.glob('*.py'))
for filename, source in AGENT_SOURCES.items():
    target = agent_directory / filename
    if not target.exists() and (filename.startswith('_') or not has_implementations):
        target.write_text(source, encoding="utf-8")
implementations = load_agents(agent_directory)
print({"implementacoes_carregadas": list(implementations)})


## 4. Setup central e nova partida

Todas as escolhas estão em **setup.json**. Edite esse arquivo no painel do Colab: `rodadas`, grupos em `tripulantes` (quantidade, papel e implementação) e `conhecimentos`. A quantidade total é a soma dos grupos. Para 50 tripulantes, por exemplo, altere a quantidade do último grupo de 1 para 46. Para 100.000 rodadas, altere apenas `rodadas`. Pode repetir qualquer implementação humana; cada participante recebe sua própria instância.

Destino: 2 × rodadas. Bônus por humano removido: rodadas ÷ humanos iniciais. A maioria considera votos válidos daquela votação. A saída continua exigindo dois bloqueadores. Esses valores são derivados pelo motor, sem novos parâmetros.

Esta célula prepara o setup se ausente e imprime a configuração carregada. Reexecutá-la cria **uma nova partida**; use a célula de retomada para conservar uma partida existente. Para salvar no Drive, configure `checkpoint` em setup.json com o caminho persistente.


In [ ]:
import json
DEFAULT_SETUP = {'rodadas': 10, 'seed': 17, 'diretorio_agentes': 'src/agent', 'checkpoint': 'estado/partida.json', 'tripulantes': [{'quantidade': 1, 'papel': 'HUMANO', 'implementacao': 'humano_alpha'}, {'quantidade': 1, 'papel': 'HUMANO', 'implementacao': 'humano_beta'}, {'quantidade': 1, 'papel': 'IA', 'implementacao': 'humanoide'}, {'quantidade': 1, 'papel': 'HUMANO', 'implementacao': 'humano_gamma'}, {'quantidade': 1, 'papel': 'HUMANO', 'implementacao': 'humano_random'}], 'conhecimentos': {'ACIDENTE': '01', 'MISSÃO': '02', 'CARGA': '03', 'PAINEL': '04', 'DISTÂNCIA': '04', 'CRISE': '05'}}
from setup_jogo import GameSetup
setup_path = Path("setup.json")
if not setup_path.exists():
    setup_path.write_text(json.dumps(DEFAULT_SETUP, ensure_ascii=False, indent=2), encoding="utf-8")
setup = GameSetup.load(setup_path)
print(json.dumps(setup.summary(), ensure_ascii=False, indent=2))
agents = setup.start()
game = agents.game
checkpoint = setup.checkpoint
print({"rodada": game.round, "fase": game.phase})


## 5. Executar e salvar entre fases

Executamos três rodadas e depois a escolha de perguntas da próxima rodada. O checkpoint conserva a fase efetivamente alcançada: perguntas, votação ou encerramento. As decisões do humano aleatório podem encerrar a partida antes dessas três rodadas.

`agents.advance(checkpoint=caminho)` salva após a fase concluída, incluindo as observações e memórias dos agentes. `agents.run_round(checkpoint=caminho)` pode salvar após cada fase da rodada. Use a sessão para executar também os hooks; a interface antiga da engine fica apenas como API de baixo nível. Salvar com frequência protege progresso, mas o histórico completo torna o arquivo maior ao longo da partida.


In [ ]:
for _ in range(3):
    if game.finished:
        break
    agents.run_round()
if not game.finished:
    agents.advance(checkpoint=checkpoint)
else:
    agents.save(checkpoint)
print({"rodada": game.round, "fase": game.phase, "checkpoint": str(checkpoint)})


## 6. Retomar

Esta célula carrega o estado salvo, inclusive a fase atual e os sorteios. Em um novo runtime, execute primeiro a definição da engine e preparação dos módulos de agentes, configure `checkpoint` e execute esta carga. Callbacks e modelos precisam ser reconectados; a memória JSON de cada instância é restaurada por `load_state`, e `on_resume` recebe somente sua visão autorizada. O início de rodada e o encerramento não são repetidos se já ocorreram antes do checkpoint.


In [ ]:
game = Game.load(checkpoint)
agents = assign_agents(game, load_agents("src/agent"))
print({"rodada_retornada": game.round, "fase_retornada": game.phase})


## 7. Continuar em lotes

Executar novamente esta célula continua a mesma partida. O lote é uma conveniência de execução, não altera regras ou N. Cada lote salva um checkpoint; para checkpoint a cada fase, passe `checkpoint=checkpoint` a `agents.run_round`.

As visões entregues automaticamente às políticas têm as últimas dez entradas de cada histórico, mas o checkpoint conserva o histórico completo. Altere `history_limit` se a política exigir outra janela, ou use `None` para todo o histórico. Em simulações longas, ler o histórico inteiro a cada ação aumenta bastante o custo.


In [ ]:
for _ in range(10000):
    if game.finished:
        break
    agents.run_round()
agents.save(checkpoint)
print({"rodada": game.round, "fase": game.phase, "encerrado": game.finished})
if game.finished:
    print(json.dumps(game.result, ensure_ascii=False, indent=2))


## 8. Inspecionar uma visão privada

O professor pode inspecionar uma visão; não encaminhe o checkpoint ou `snapshot()` aos agentes. `history_limit=None` permite consultar o histórico completo autorizado de um agente. O resultado distingue motivo de encerramento e objetivos. Por determinação do jogo, direção final ao planeta significa entrega observável da carga; direção final à estação espacial significa retorno e sobrevivência humana. Esses resultados são registrados no encerramento, sem executar um trajeto posterior e sem definir um vencedor único.


In [ ]:
view_01 = game.view("01")
print({"id": view_01["id"], "respostas_recebidas": view_01["history_sizes"]["received"],
       "revelacoes_recebidas": view_01["history_sizes"]["revelations"]})
